# Audio Deepfake Domain Generalization

Compare ERM, IRM, GroupDRO, and CORAL with RawNet2 on held-out French speech generators.

Run in Google Colab with the paired M-AILABS and MLAAD archives in Drive. Update dataset and output paths as needed. Training uses four selected domain splits.

## Google Drive

Mount Drive to access the dataset archives.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Dataset archives

Copy the paired M-AILABS and MLAAD French archives to the Colab runtime.

In [ ]:
!cp /content/drive/MyDrive/mailabs_fr_paired_only.tar /content/
!cp /content/drive/MyDrive/mlaad_fr.tar.gz /content/

print("Archives copied from Google Drive to local Colab.")

Archives copied from Google Drive to local Colab.


## Extract audio

In [ ]:
# Extract paired-only M-AILABS real files
!tar -xf /content/mailabs_fr_paired_only.tar -C /content/

# Extract MLAAD French fake files
!mkdir -p /content/mlaad_fr
!tar -xzf /content/mlaad_fr.tar.gz -C /content/mlaad_fr

print("Both archives extracted.")

Both archives extracted.


## RawNet2 implementation

Clone the official anti-spoofing implementation.

In [ ]:
!git clone https://github.com/eurecom-asp/rawnet2-antispoofing.git /content/rawnet2_official

!ls /content/rawnet2_official

Cloning into '/content/rawnet2_official'...
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 98 (delta 8), reused 0 (delta 0), pack-reused 78 (from 1)
Receiving objects: 100% (98/98), 275.91 MiB | 40.69 MiB/s, done.
Resolving deltas: 100% (30/30), done.
data_utils_LA.py	       model_config_RawNet2.yaml  SVM_fusion
LFCC_high_resolution_baseline  model.py			  tDCF_python
LICENSE			       README.md
main.py			       requirements.txt


## Shared initialization

Load or save the initial RawNet2 weights used across experiments.

In [ ]:
import os
import sys
import yaml
import torch

sys.path.append("/content/rawnet2_official")

from model import RawNet

WEIGHTS_PATH = "/content/rawnet_model_initial_weights.pth"

with open("/content/rawnet2_official/model_config_RawNet2.yaml", "r") as f:
    config = yaml.safe_load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = RawNet(config["model"], device).to(device)

try:
    if not os.path.exists(WEIGHTS_PATH):
        raise FileNotFoundError(f"Weights file not found: {WEIGHTS_PATH}")

    state_dict = torch.load(WEIGHTS_PATH, map_location=device)
    model.load_state_dict(state_dict, strict=True)

    print(f"Loaded existing initial weights from: {WEIGHTS_PATH}")

except Exception as e:
    torch.save(model.state_dict(), WEIGHTS_PATH)

    print("Could not load existing initial weights.")
    print(f"Reason: {repr(e)}")
    print(f"Saved currently initialized model weights to: {WEIGHTS_PATH}")

print(model)

Could not load existing initial weights.
Reason: RuntimeError('PytorchStreamReader failed reading zip archive: failed finding central directory')
Saved currently initialized model weights to: /content/rawnet_model_initial_weights.pth
RawNet(
  (Sinc_conv): SincConv()
  (first_bn): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (selu): SELU(inplace=True)
  (block0): Sequential(
    (0): Residual_block(
      (lrelu): LeakyReLU(negative_slope=0.3)
      (conv1): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (mp): MaxPool1d(kernel_size=3, stride=3, padding=0, dilation=1, ceil_mode=False)
    )
  )
  (block1): Sequential(
    (0): Residual_block(
      (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (lrelu): LeakyReLU(n

## Model sanity check

Check the output shape for two four-second waveforms.

In [ ]:
x = torch.randn(2, 64000).to(device)  # 4 sec at 16 kHz
with torch.no_grad():
    y = model(x)

print(y.shape)

torch.Size([2, 2])


## Dataset indexing

Configure the six synthesis domains and index paired real recordings.

In [ ]:
from pathlib import Path
from itertools import combinations
from sklearn.model_selection import train_test_split
import pandas as pd

# =========================
# CONFIG
# =========================

MLAAD_FR_ROOT = Path("/content/mlaad_fr/fr")
MAILABS_ROOT = Path("/content/mailabs_fr_paired_only")
OUT_ROOT = Path("/content/dg_splits_utterance_safe")

SEED = 42
VAL_RATIO = 0.20

SELECTED_GENERATORS = [
    "facebook_mms-tts-fra",
    "griffin_lim",
    "tts_models_fr_css10_vits",
    "tts_models_fr_mai_tacotron2-DDC",
    "tts_models_multilingual_multi-dataset_bark",
    "tts_models_multilingual_multi-dataset_xtts_v1.1",
]

# =========================
# CHECKS
# =========================

assert MLAAD_FR_ROOT.exists(), f"MLAAD French root not found: {MLAAD_FR_ROOT}"
assert MAILABS_ROOT.exists(), f"M-AILABS root not found: {MAILABS_ROOT}"

OUT_ROOT.mkdir(parents=True, exist_ok=True)

for gen in SELECTED_GENERATORS:
    assert (MLAAD_FR_ROOT / gen).exists(), f"Missing generator folder: {gen}"

# =========================
# INDEX REAL MAILABS FILES
# =========================

print("Indexing M-AILABS real wavs...")

real_by_name = {}
duplicate_real_names = []

for p in MAILABS_ROOT.rglob("*.wav"):
    if p.name in real_by_name:
        duplicate_real_names.append(p.name)
    else:
        real_by_name[p.name] = str(p)

print("Unique M-AILABS real wav basenames:", len(real_by_name))
print("Duplicate M-AILABS basenames:", len(duplicate_real_names))


Indexing M-AILABS real wavs...
Unique M-AILABS real wav basenames: 7686
Duplicate M-AILABS basenames: 0


## Utterance-disjoint splits

Create all 15 four-source/two-target domain splits. Exclude target utterances from source data and check for utterance and real-waveform overlap.

In [ ]:
# =========================
# BUILD PAIRED METADATA
# =========================

rows = []
missing_real = []

for gen in SELECTED_GENERATORS:
    fake_files = sorted((MLAAD_FR_ROOT / gen).rglob("*.wav"))

    print(f"{gen}: {len(fake_files)} fake wavs")

    for fake_path in fake_files:
        utterance_id = fake_path.name

        if utterance_id not in real_by_name:
            missing_real.append((gen, utterance_id))
            continue

        # Spoof row
        rows.append({
            "audio_path": str(fake_path),
            "label": 1,
            "spoof_generator": gen,
            "paired_domain": gen,
            "utterance_id": utterance_id,
        })

        # Paired bonafide row, assigned to the same paired domain
        rows.append({
            "audio_path": real_by_name[utterance_id],
            "label": 0,
            "spoof_generator": "none",
            "paired_domain": gen,
            "utterance_id": utterance_id,
        })

assert len(missing_real) == 0, f"Missing real pairs. First examples: {missing_real[:20]}"

all_df = pd.DataFrame(rows)

all_metadata_path = OUT_ROOT / "all_metadata.csv"
all_df.to_csv(all_metadata_path, index=False)

print("\nSaved all metadata:", all_metadata_path)
print("Total rows:", len(all_df))
print("\nCounts by paired_domain and label:")
print(all_df.groupby(["paired_domain", "label"]).size())

# =========================
# UTTERANCE-SAFE SPLITS
# =========================

test_domain_pairs = list(combinations(SELECTED_GENERATORS, 2))

print("\nTotal 4-train / 2-test splits:", len(test_domain_pairs))

summary_rows = []

for split_idx, test_domains_tuple in enumerate(test_domain_pairs):
    test_domains = list(test_domains_tuple)
    train_domains = [g for g in SELECTED_GENERATORS if g not in test_domains]

    split_dir = OUT_ROOT / f"split_{split_idx:02d}"
    split_dir.mkdir(parents=True, exist_ok=True)

    # Candidate target rows: all rows from held-out domains
    raw_test_df = all_df[all_df["paired_domain"].isin(test_domains)].copy()

    # Any utterance appearing in the target domains is banned from source train/val
    test_utterance_ids = set(raw_test_df["utterance_id"].unique())

    # Source rows: train domains only, but remove any utterance that appears in test
    source_df = all_df[
        (all_df["paired_domain"].isin(train_domains)) &
        (~all_df["utterance_id"].isin(test_utterance_ids))
    ].copy()

    # Test rows: keep held-out domains fully held out
    test_df = raw_test_df.copy()

    # Split source train/val by utterance_id, not by row
    source_utterances = sorted(source_df["utterance_id"].unique())

    train_utterances, val_utterances = train_test_split(
        source_utterances,
        test_size=VAL_RATIO,
        random_state=SEED,
        shuffle=True
    )

    train_utterances = set(train_utterances)
    val_utterances = set(val_utterances)

    train_df = source_df[source_df["utterance_id"].isin(train_utterances)].copy()
    val_df = source_df[source_df["utterance_id"].isin(val_utterances)].copy()

    train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    val_df = val_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    test_df = test_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

    train_df["split"] = "train"
    val_df["split"] = "val"
    test_df["split"] = "test"

    # Leakage checks
    train_utts = set(train_df["utterance_id"])
    val_utts = set(val_df["utterance_id"])
    test_utts = set(test_df["utterance_id"])

    assert train_utts.isdisjoint(val_utts), "Train/val utterance leakage found"
    assert train_utts.isdisjoint(test_utts), "Train/test utterance leakage found"
    assert val_utts.isdisjoint(test_utts), "Val/test utterance leakage found"

    # Exact real waveform leakage checks
    train_real_paths = set(train_df[train_df["label"] == 0]["audio_path"])
    val_real_paths = set(val_df[val_df["label"] == 0]["audio_path"])
    test_real_paths = set(test_df[test_df["label"] == 0]["audio_path"])

    assert train_real_paths.isdisjoint(val_real_paths), "Train/val real waveform leakage found"
    assert train_real_paths.isdisjoint(test_real_paths), "Train/test real waveform leakage found"
    assert val_real_paths.isdisjoint(test_real_paths), "Val/test real waveform leakage found"

    train_path = split_dir / "main_train.csv"
    val_path = split_dir / "main_val.csv"
    test_path = split_dir / "main_test.csv"
    combined_path = split_dir / "main_split.csv"

    train_df.to_csv(train_path, index=False)
    val_df.to_csv(val_path, index=False)
    test_df.to_csv(test_path, index=False)

    combined_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    combined_df.to_csv(combined_path, index=False)

    dropped_source_rows = len(
        all_df[
            (all_df["paired_domain"].isin(train_domains)) &
            (all_df["utterance_id"].isin(test_utterance_ids))
        ]
    )

    summary_rows.append({
        "split": f"split_{split_idx:02d}",
        "train_domains": "|".join(train_domains),
        "test_domains": "|".join(test_domains),
        "train_rows": len(train_df),
        "val_rows": len(val_df),
        "test_rows": len(test_df),
        "dropped_source_rows_due_to_test_utterance_overlap": dropped_source_rows,
        "train_utterances": len(train_utts),
        "val_utterances": len(val_utts),
        "test_utterances": len(test_utts),
    })

    print(f"\nSplit {split_idx:02d}")
    print("Train domains:", train_domains)
    print("Test domains:", test_domains)
    print("Dropped source rows due to test utterance overlap:", dropped_source_rows)

    print("\nTrain counts:")
    print(train_df.groupby(["paired_domain", "label"]).size())

    print("\nVal counts:")
    print(val_df.groupby(["paired_domain", "label"]).size())

    print("\nTest counts:")
    print(test_df.groupby(["paired_domain", "label"]).size())

summary_df = pd.DataFrame(summary_rows)
summary_path = OUT_ROOT / "split_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("\nDone.")
print("Output root:", OUT_ROOT)
print("Summary saved:", summary_path)

facebook_mms-tts-fra: 1000 fake wavs
griffin_lim: 1000 fake wavs
tts_models_fr_css10_vits: 1000 fake wavs
tts_models_fr_mai_tacotron2-DDC: 1000 fake wavs
tts_models_multilingual_multi-dataset_bark: 1000 fake wavs
tts_models_multilingual_multi-dataset_xtts_v1.1: 1000 fake wavs

Saved all metadata: /content/dg_splits_utterance_safe/all_metadata.csv
Total rows: 12000

Counts by paired_domain and label:
paired_domain                                    label
facebook_mms-tts-fra                             0        1000
                                                 1        1000
griffin_lim                                      0        1000
                                                 1        1000
tts_models_fr_css10_vits                         0        1000
                                                 1        1000
tts_models_fr_mai_tacotron2-DDC                  0        1000
                                                 1        1000
tts_models_multilingual_multi-dataset_

### Export splits

In [ ]:
import os
from google.colab import files

directory_to_archive = "/content/dg_splits_utterance_safe"
archive_name = "dg_splits_utterance_safe.tar.gz"

# Create the tar.gz archive
!tar -czvf {archive_name} {directory_to_archive}

# Download the archive
files.download(archive_name)

## ERM baseline

In [ ]:
import os
import random
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchaudio
from tqdm.auto import tqdm


# ============================================================
# Config
# ============================================================

SPLITS_ROOT = Path("/content/dg_splits_utterance_safe")
INITIAL_WEIGHTS_PATH = "/content/rawnet_model_initial_weights.pth"
RESULTS_ROOT = Path("/content/ERM_results")
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

SELECTED_SPLITS = [
    "split_00",
    "split_09",
    "split_14",
    "split_12",
]

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TARGET_SR = 16000
CLIP_SECONDS = 4
CLIP_LEN = TARGET_SR * CLIP_SECONDS  # 64000

# L4 Colab Pro should usually handle this comfortably for 4s RawNet inputs.
# If you hit CUDA OOM, lower to 48 or 32. If memory is still low, try 96/128.
BATCH_SIZE = 128
NUM_WORKERS = 4
EPOCHS = 10
LR = 1e-5
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 5.0


# ============================================================
# Reproducibility
# ============================================================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Keeps runs more reproducible. If a model op has no deterministic
    # implementation, PyTorch will warn instead of crashing the experiment.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def seed_worker(worker_id):
    worker_seed = (torch.initial_seed() + worker_id) % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


seed_everything(SEED)


# ============================================================
# Validation helpers
# ============================================================

def require_file(path, description):
    if not os.path.isfile(path):
        raise FileNotFoundError(f"{description} not found: {path}")


def validate_split_dataframe(df, csv_path, split_name):
    required_cols = {"audio_path", "label"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns in {csv_path}: {sorted(missing)}")

    if len(df) == 0:
        raise ValueError(f"{split_name} split is empty: {csv_path}")

    invalid_labels = sorted(set(df["label"].dropna().unique()) - {0, 1})
    if invalid_labels:
        raise ValueError(
            f"{split_name} split has labels outside {{0, 1}}: {invalid_labels}"
        )

    label_count = df["label"].nunique(dropna=True)
    if label_count != 2:
        raise ValueError(
            f"{split_name} split must contain both classes 0 and 1, "
            f"but found labels: {sorted(df['label'].dropna().unique())}"
        )


def validate_no_missing_paths(df, csv_path, split_name, out_dir):
    exists_mask = df["audio_path"].apply(
        lambda p: isinstance(p, str) and os.path.isfile(p)
    )
    missing_df = df.loc[~exists_mask].copy()

    if len(missing_df) == 0:
        return df.reset_index(drop=True)

    report_path = os.path.join(out_dir, f"{split_name}_missing_audio_paths.csv")
    missing_df.to_csv(report_path, index=False)

    raise FileNotFoundError(
        f"{split_name} split has {len(missing_df)} missing audio files. "
        f"Report saved to: {report_path}"
    )


def validate_loader_not_empty(loader, split_name):
    if len(loader.dataset) == 0:
        raise ValueError(f"{split_name} DataLoader has an empty dataset")


def discover_split_dirs(splits_root, selected_splits=None):
    if selected_splits is None:
        split_dirs = sorted(
            p for p in splits_root.glob("split_*")
            if p.is_dir()
        )
    else:
        split_dirs = [splits_root / split_name for split_name in selected_splits]

    if not split_dirs:
        raise FileNotFoundError(f"No split directories selected under: {splits_root}")

    missing_dirs = [str(p) for p in split_dirs if not p.is_dir()]
    if missing_dirs:
        raise FileNotFoundError(f"Selected split directories not found: {missing_dirs}")

    for split_dir in split_dirs:
        require_file(split_dir / "main_train.csv", f"{split_dir.name} train CSV")
        require_file(split_dir / "main_val.csv", f"{split_dir.name} val CSV")
        require_file(split_dir / "main_test.csv", f"{split_dir.name} test CSV")

    return split_dirs


# ============================================================
# Dataset
# ============================================================

class AudioCSVDataset(Dataset):
    def __init__(
        self,
        csv_path,
        target_sr=16000,
        clip_len=64000,
        train=True,
        split_name="split",
        out_dir=None,
    ):
        require_file(csv_path, f"{split_name} CSV")

        self.df = pd.read_csv(csv_path)
        self.target_sr = target_sr
        self.clip_len = clip_len
        self.train = train
        self.split_name = split_name
        self.out_dir = out_dir

        validate_split_dataframe(self.df, csv_path, split_name)

        before = len(self.df)
        self.df = validate_no_missing_paths(self.df, csv_path, split_name, out_dir)
        after = len(self.df)

        print(f"\nLoaded: {csv_path}")
        print(f"Rows before path check: {before}")
        print(f"Rows after path check:  {after}")
        print("Label counts:")
        print(self.df["label"].value_counts().sort_index())

        if "paired_domain" in self.df.columns:
            print("Paired domain counts:")
            print(self.df.groupby(["paired_domain", "label"]).size())

    def __len__(self):
        return len(self.df)

    def _load_audio(self, path):
        try:
            wav, sr = torchaudio.load(path)
        except Exception as exc:
            raise RuntimeError(f"Failed to load audio file: {path}") from exc

        if wav.numel() == 0:
            raise RuntimeError(f"Audio file is empty: {path}")

        # Convert stereo/multi-channel to mono
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)

        # Resample to RawNet expected sampling rate
        if sr != self.target_sr:
            wav = torchaudio.functional.resample(wav, sr, self.target_sr)

        wav = wav.squeeze(0)  # [T]

        if not torch.isfinite(wav).all():
            raise RuntimeError(f"Audio contains NaN or Inf values: {path}")

        # Simple peak normalization
        max_val = wav.abs().max()
        if max_val > 0:
            wav = wav / max_val

        return wav

    def _crop_or_pad(self, wav):
        T = wav.numel()

        if T == self.clip_len:
            return wav

        if T > self.clip_len:
            if self.train:
                start = random.randint(0, T - self.clip_len)
            else:
                start = (T - self.clip_len) // 2
            return wav[start:start + self.clip_len]

        pad_len = self.clip_len - T
        return torch.nn.functional.pad(wav, (0, pad_len))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        wav = self._load_audio(row["audio_path"])
        wav = self._crop_or_pad(wav)

        label = torch.tensor(int(row["label"]), dtype=torch.long)

        item = {
            "waveform": wav,
            "label": label,
            "audio_path": row["audio_path"],
        }

        if "paired_domain" in self.df.columns:
            item["paired_domain"] = row["paired_domain"]

        if "spoof_generator" in self.df.columns:
            item["spoof_generator"] = row["spoof_generator"]

        if "utterance_id" in self.df.columns:
            item["utterance_id"] = row["utterance_id"]

        return item


def collate_fn(batch):
    if len(batch) == 0:
        raise ValueError("Received an empty batch")

    waveforms = torch.stack([x["waveform"] for x in batch], dim=0)  # [B, 64000]
    labels = torch.stack([x["label"] for x in batch], dim=0)  # [B]

    out = {
        "waveform": waveforms,
        "label": labels,
        "audio_path": [x["audio_path"] for x in batch],
    }

    for key in ["paired_domain", "spoof_generator", "utterance_id"]:
        if key in batch[0]:
            out[key] = [x[key] for x in batch]

    return out


def make_loaders(split_dir, out_dir):
    train_csv = split_dir / "main_train.csv"
    val_csv = split_dir / "main_val.csv"
    test_csv = split_dir / "main_test.csv"

    train_ds = AudioCSVDataset(
        train_csv, TARGET_SR, CLIP_LEN, train=True,
        split_name=f"{split_dir.name}_train", out_dir=out_dir
    )
    val_ds = AudioCSVDataset(
        val_csv, TARGET_SR, CLIP_LEN, train=False,
        split_name=f"{split_dir.name}_val", out_dir=out_dir
    )
    test_ds = AudioCSVDataset(
        test_csv, TARGET_SR, CLIP_LEN, train=False,
        split_name=f"{split_dir.name}_test", out_dir=out_dir
    )

    loader_generator = torch.Generator()
    loader_generator.manual_seed(SEED)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
        collate_fn=collate_fn,
        drop_last=False,
        worker_init_fn=seed_worker,
        generator=loader_generator,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
        collate_fn=collate_fn,
        drop_last=False,
        worker_init_fn=seed_worker,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
        collate_fn=collate_fn,
        drop_last=False,
        worker_init_fn=seed_worker,
    )

    validate_loader_not_empty(train_loader, f"{split_dir.name} train")
    validate_loader_not_empty(val_loader, f"{split_dir.name} val")
    validate_loader_not_empty(test_loader, f"{split_dir.name} test")

    return train_loader, val_loader, test_loader


# ============================================================
# Model setup
# ============================================================
# Assumption: model has already been constructed.
# This script reloads /content/rawnet_model_initial_weights.pth before each
# split so every ERM run starts from the same initial model.

if "model" not in globals():
    raise NameError(
        "Expected a pre-constructed variable named `model`, but none was found. "
        "Create your RawNet/model object before running this training cell."
    )

require_file(INITIAL_WEIGHTS_PATH, "Initial model weights")


def extract_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in ["model_state_dict", "state_dict", "model"]:
            if key in checkpoint and isinstance(checkpoint[key], dict):
                return checkpoint[key]

        if checkpoint and all(torch.is_tensor(v) for v in checkpoint.values()):
            return checkpoint

    raise ValueError(
        "Could not find a model state_dict in the initial weights file. "
        "Expected either a raw state_dict or a dict with model_state_dict/state_dict/model."
    )


def strip_module_prefix(state_dict):
    if not any(k.startswith("module.") for k in state_dict.keys()):
        return state_dict

    return {
        k.replace("module.", "", 1): v
        for k, v in state_dict.items()
    }


initial_checkpoint = torch.load(INITIAL_WEIGHTS_PATH, map_location="cpu")
initial_state_dict = strip_module_prefix(extract_state_dict(initial_checkpoint))

model = model.to(DEVICE)
model.float()
criterion = nn.CrossEntropyLoss()


def reset_model_to_initial_weights(model):
    model.load_state_dict(initial_state_dict, strict=True)
    model.to(DEVICE)
    model.float()


def forward_model(model, waveforms):
    """
    waveforms: [B, 64000]
    returns logits: [B, 2]
    """
    if waveforms.ndim != 2 or waveforms.shape[1] != CLIP_LEN:
        raise ValueError(f"Expected waveforms shape [B, {CLIP_LEN}], got {waveforms.shape}")

    logits = model(waveforms)

    if logits.ndim != 2 or logits.shape[1] != 2:
        raise ValueError(f"Expected logits shape [B, 2], got {logits.shape}")

    if not torch.isfinite(logits).all():
        raise RuntimeError("Model produced NaN or Inf logits")

    return logits


def compute_loss_and_scores(logits, labels):
    """
    labels:
        0 = bonafide
        1 = spoof

    logits:
        [B, 2]

    score:
        probability of class 1, i.e. spoof/fake
    """
    loss = criterion(logits, labels)
    scores = torch.softmax(logits, dim=1)[:, 1]
    return loss, scores


# ============================================================
# Metrics
# ============================================================

def compute_accuracy(labels, scores, threshold=0.5):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores).astype(float)

    if len(labels) == 0:
        return float("nan")

    preds = (scores >= threshold).astype(int)
    return float((preds == labels).mean())


def compute_eer(labels, scores):
    """
    labels:
        0 = bonafide
        1 = spoof

    scores:
        higher score = more likely spoof
    """
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores).astype(float)

    if len(labels) == 0:
        return float("nan")

    finite_mask = np.isfinite(scores)
    labels = labels[finite_mask]
    scores = scores[finite_mask]

    P = np.sum(labels == 1)
    N = np.sum(labels == 0)

    if P == 0 or N == 0:
        return float("nan")

    # Include boundary thresholds so all-negative and all-positive predictions
    # are represented. This keeps the metric stable for tiny validation sets.
    thresholds = np.r_[np.inf, np.unique(scores)[::-1], -np.inf]

    fars = []
    frrs = []

    for thr in thresholds:
        preds = (scores >= thr).astype(int)

        fp = np.sum((preds == 1) & (labels == 0))
        fn = np.sum((preds == 0) & (labels == 1))

        far = fp / N
        frr = fn / P

        fars.append(far)
        frrs.append(frr)

    fars = np.asarray(fars)
    frrs = np.asarray(frrs)

    idx = np.argmin(np.abs(fars - frrs))
    eer = (fars[idx] + frrs[idx]) / 2.0

    return float(eer)


@torch.no_grad()
def evaluate(model, loader, split_name="val"):
    model.eval()

    all_labels = []
    all_scores = []
    all_domains = []
    all_paths = []

    total_loss = 0.0
    total_n = 0

    # Keep evaluation intentionally quiet. Training already owns the visible
    # progress bar, and nested tqdm bars are easy to make messy in notebooks.
    for batch in loader:
        waveforms = batch["waveform"].to(DEVICE, non_blocking=True).float()
        labels = batch["label"].to(DEVICE, non_blocking=True).long()

        logits = forward_model(model, waveforms)
        loss, scores = compute_loss_and_scores(logits, labels)

        if not torch.isfinite(loss):
            raise RuntimeError(f"{split_name} evaluation produced invalid loss: {loss.item()}")

        bs = labels.shape[0]
        total_loss += loss.item() * bs
        total_n += bs

        all_labels.extend(labels.detach().cpu().numpy().tolist())
        all_scores.extend(scores.detach().cpu().numpy().tolist())
        all_paths.extend(batch["audio_path"])

        if "paired_domain" in batch:
            all_domains.extend(batch["paired_domain"])
        else:
            all_domains.extend(["unknown"] * bs)

    if total_n == 0:
        raise ValueError(f"{split_name} evaluation received zero samples")

    avg_loss = total_loss / total_n
    acc = compute_accuracy(all_labels, all_scores)
    eer = compute_eer(all_labels, all_scores)

    result = {
        "split": split_name,
        "loss": avg_loss,
        "accuracy": acc,
        "eer": eer,
        "num_samples": total_n,
    }

    pred_df = pd.DataFrame({
        "audio_path": all_paths,
        "label": all_labels,
        "score": all_scores,
        "paired_domain": all_domains,
    })

    per_domain_rows = []

    for domain, g in pred_df.groupby("paired_domain"):
        per_domain_rows.append({
            "split": split_name,
            "paired_domain": domain,
            "num_samples": len(g),
            "accuracy": compute_accuracy(g["label"].values, g["score"].values),
            "eer": compute_eer(g["label"].values, g["score"].values),
        })

    per_domain_df = pd.DataFrame(per_domain_rows)

    return result, per_domain_df, pred_df


# ============================================================
# Training
# ============================================================

def run_one_split(split_dir):
    split_name = split_dir.name
    out_dir = RESULTS_ROOT / split_name
    out_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 80)
    print(f"Starting ERM run: {split_name}")
    print(f"Outputs: {out_dir}")
    print("=" * 80)

    seed_everything(SEED)
    reset_model_to_initial_weights(model)

    train_loader, val_loader, test_loader = make_loaders(split_dir, out_dir)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    best_val_eer = float("inf")
    best_val_loss = float("inf")
    best_epoch = None
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()

        running_loss = 0.0
        running_n = 0
        skipped_batches = 0

        train_pbar = tqdm(
            train_loader,
            desc=f"{split_name} | Epoch {epoch:02d}/{EPOCHS}",
            leave=True,
            dynamic_ncols=True,
        )

        for batch in train_pbar:
            waveforms = batch["waveform"].to(DEVICE, non_blocking=True).float()
            labels = batch["label"].to(DEVICE, non_blocking=True).long()

            optimizer.zero_grad(set_to_none=True)

            logits = forward_model(model, waveforms)
            loss, scores = compute_loss_and_scores(logits, labels)

            if not torch.isfinite(loss):
                skipped_batches += 1
                warnings.warn(f"Skipping batch due to invalid loss: {loss.item()}")
                continue

            loss.backward()

            if GRAD_CLIP is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

            optimizer.step()

            bs = labels.shape[0]
            running_loss += loss.item() * bs
            running_n += bs

            train_pbar.set_postfix({
                "train_loss": f"{running_loss / max(running_n, 1):.4f}",
                "skipped": skipped_batches,
            })

        if running_n == 0:
            raise RuntimeError(
                f"{split_name} epoch {epoch} completed with zero usable training samples. "
                f"Skipped batches: {skipped_batches}"
            )

        train_loss = running_loss / running_n

        val_result, val_per_domain_df, val_pred_df = evaluate(
            model,
            val_loader,
            split_name="val"
        )

        print(
            f"\n{split_name} | Epoch {epoch:02d} done | "
            f"Train loss: {train_loss:.4f} | "
            f"Val loss: {val_result['loss']:.4f} | "
            f"Val acc: {val_result['accuracy']:.4f} | "
            f"Val EER: {val_result['eer']:.4f} | "
            f"Skipped batches: {skipped_batches}\n"
        )

        epoch_record = {
            "split_dir": str(split_dir),
            "run": split_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_result["loss"],
            "val_accuracy": val_result["accuracy"],
            "val_eer": val_result["eer"],
            "val_num_samples": val_result["num_samples"],
            "skipped_batches": skipped_batches,
        }

        history.append(epoch_record)

        val_eer = val_result["eer"]
        val_loss = val_result["loss"]

        # Primary selection is still validation EER. If EER is unavailable for an
        # unexpected reason, fall back to validation loss so a checkpoint exists.
        if math.isfinite(val_eer):
            is_new_best = val_eer < best_val_eer
        else:
            is_new_best = not math.isfinite(best_val_eer) and val_loss < best_val_loss

        if is_new_best:
            best_val_eer = val_eer
            best_val_loss = val_loss
            best_epoch = epoch

            ckpt_path = out_dir / "best_erm.pt"

            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_eer": best_val_eer,
                "best_val_loss": best_val_loss,
                "config": {
                    "split_dir": str(split_dir),
                    "initial_weights_path": INITIAL_WEIGHTS_PATH,
                    "target_sr": TARGET_SR,
                    "clip_seconds": CLIP_SECONDS,
                    "clip_len": CLIP_LEN,
                    "batch_size": BATCH_SIZE,
                    "lr": LR,
                    "weight_decay": WEIGHT_DECAY,
                    "epochs": EPOCHS,
                    "seed": SEED,
                }
            }, ckpt_path)

            val_pred_df.to_csv(out_dir / "best_val_predictions.csv", index=False)
            val_per_domain_df.to_csv(out_dir / "best_val_per_domain.csv", index=False)

            print(f"Saved new best checkpoint: {ckpt_path}")
            print(f"Best val EER: {best_val_eer:.4f}")

    pd.DataFrame(history).to_csv(out_dir / "training_history.csv", index=False)

    best_ckpt_path = out_dir / "best_erm.pt"

    if not best_ckpt_path.is_file():
        raise RuntimeError(
            f"No best checkpoint was saved for {split_name}. Check that validation "
            "contains both classes and produces finite metrics."
        )

    best_ckpt = torch.load(best_ckpt_path, map_location=DEVICE)

    model.load_state_dict(best_ckpt["model_state_dict"])
    model.to(DEVICE)
    model.float()

    test_result, test_per_domain_df, test_pred_df = evaluate(
        model,
        test_loader,
        split_name="test"
    )

    test_result["run"] = split_name
    test_result["split_dir"] = str(split_dir)
    test_result["best_epoch"] = best_epoch
    test_result["best_val_eer"] = best_val_eer
    test_result["best_val_loss"] = best_val_loss
    test_result["batch_size"] = BATCH_SIZE
    test_result["initial_weights_path"] = INITIAL_WEIGHTS_PATH

    print(f"\n{split_name} | Final Test Result:")
    print(test_result)

    print(f"\n{split_name} | Test Per Domain:")
    print(test_per_domain_df)

    test_pred_df.to_csv(out_dir / "test_predictions.csv", index=False)
    test_per_domain_df.to_csv(out_dir / "test_per_domain.csv", index=False)

    with open(out_dir / "test_result.json", "w") as f:
        json.dump(test_result, f, indent=2)

    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    print(f"\nSaved outputs to: {out_dir}")
    return test_result


# ============================================================
# Run all splits
# ============================================================

split_dirs = discover_split_dirs(SPLITS_ROOT, SELECTED_SPLITS)
print(f"Found {len(split_dirs)} split directories under {SPLITS_ROOT}")
print(f"Selected splits: {', '.join(p.name for p in split_dirs)}")
print(f"Using batch size: {BATCH_SIZE}")
print(f"Initial weights: {INITIAL_WEIGHTS_PATH}")
print(f"Results root: {RESULTS_ROOT}")

all_results = []

for split_dir in split_dirs:
    split_result = run_one_split(split_dir)
    all_results.append(split_result)

all_results_df = pd.DataFrame(all_results)
all_results_path = RESULTS_ROOT / "all_splits_test_results.csv"
all_results_df.to_csv(all_results_path, index=False)

summary = {
    "num_splits": len(all_results_df),
    "mean_test_accuracy": float(all_results_df["accuracy"].mean()),
    "std_test_accuracy": float(all_results_df["accuracy"].std(ddof=0)),
    "mean_test_eer": float(all_results_df["eer"].mean()),
    "std_test_eer": float(all_results_df["eer"].std(ddof=0)),
    "results_csv": str(all_results_path),
}

with open(RESULTS_ROOT / "all_splits_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nAll splits complete.")
print(all_results_df)
print("\nAggregate summary:")
print(summary)
print("Saved aggregate results:", all_results_path)




Found 4 split directories under /content/dg_splits_utterance_safe
Selected splits: split_00, split_09, split_14, split_12
Using batch size: 128
Initial weights: /content/rawnet_model_initial_weights.pth
Results root: /content/ERM_results

Starting ERM run: split_00
Outputs: /content/ERM_results/split_00

Loaded: /content/dg_splits_utterance_safe/split_00/main_train.csv
Rows before path check: 6268
Rows after path check:  6268
Label counts:
label
0    3134
1    3134
Name: count, dtype: int64
Paired domain counts:
paired_domain                                    label
tts_models_fr_css10_vits                         0        781
                                                 1        781
tts_models_fr_mai_tacotron2-DDC                  0        786
                                                 1        786
tts_models_multilingual_multi-dataset_bark       0        794
                                                 1        794
tts_models_multilingual_multi-dataset_xtts_v1.1  0     

split_00 | Epoch 01/10:   0%|          | 0/49 [00:00<?, ?it/s]


split_00 | Epoch 01 done | Train loss: 0.5194 | Val loss: 2.5785 | Val acc: 0.5000 | Val EER: 0.3671 | Skipped batches: 0

Saved new best checkpoint: /content/ERM_results/split_00/best_erm.pt
Best val EER: 0.3671


split_00 | Epoch 02/10:   0%|          | 0/49 [00:00<?, ?it/s]


split_00 | Epoch 02 done | Train loss: 0.2894 | Val loss: 1.9280 | Val acc: 0.5212 | Val EER: 0.1630 | Skipped batches: 0

Saved new best checkpoint: /content/ERM_results/split_00/best_erm.pt
Best val EER: 0.1630


...

split_12 | Epoch 10/10:   0%|          | 0/49 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x789c411dd260>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x789c411dd260>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16


split_12 | Epoch 10 done | Train loss: 0.0957 | Val loss: 0.1280 | Val acc: 0.9493 | Val EER: 0.0482 | Skipped batches: 0



Exception ignored in: Exception ignored in: Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x789c411dd260><function _MultiProcessingDataLoaderIter.__del__ at 0x789c411dd260><function _MultiProcessingDataLoaderIter.__del__ at 0x789c411dd260><function _MultiProcessingDataLoaderIter.__del__ at 0x789c411dd260>



Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
                self._shutdown_workers()self._shutdown_workers()self._shutdown_workers()


split_12 | Final Test Result:
{'split': 'test', 'loss': 1.9444884071350097, 'accuracy': 0.5965, 'eer': 0.332, 'num_samples': 4000, 'run': 'split_12', 'split_dir': '/content/dg_splits_utterance_safe/split_12', 'best_epoch': 8, 'best_val_eer': 0.03929024081115336, 'best_val_loss': 0.10665635703218905, 'batch_size': 128, 'initial_weights_path': '/content/rawnet_model_initial_weights.pth'}

split_12 | Test Per Domain:
  split                               paired_domain  num_samples  accuracy  \
0  test             tts_models_fr_mai_tacotron2-DDC         2000    0.5035   
1  test  tts_models_multilingual_multi-dataset_bark         2000    0.6895   

     eer  
0  0.378  
1  0.267  

Saved outputs to: /content/ERM_results/split_12

All splits complete.
  split      loss  accuracy     eer  num_samples       run  \
0  test  1.515319   0.66375  0.2930         4000  split_00   
1  test  1.645737   0.50850  0.4560         4000  split_09   
2  test  2.348197   0.58025  0.3525         4000  split_

### Export ERM results

Archive results without model weights. The helper is reused below.

In [ ]:
import shutil
import os
from google.colab import files

def prepare_and_download(source_dir, dest_dir, archive_name):
    def ignore_weights(directory, filenames):
        return [f for f in filenames if f.endswith(('.pt', '.pth'))]

    os.makedirs(dest_dir, exist_ok=True)
    shutil.copytree(source_dir, dest_dir, ignore=ignore_weights, dirs_exist_ok=True)
    !tar -czf {archive_name} -C {os.path.dirname(dest_dir)} {os.path.basename(dest_dir)}
    files.download(archive_name)
    print(f"Archived {source_dir} (sans weights) and started download.")

prepare_and_download('/content/ERM_results', '/content/ERM_results_clean', 'ERM_results_clean.tar.gz')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Archived /content/ERM_results (sans weights) and started download.


## Invariant Risk Minimization (IRM)

In [ ]:
import os
import random
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler

import torchaudio
from tqdm.auto import tqdm


# ============================================================
# Config
# ============================================================

SPLITS_ROOT = Path("/content/dg_splits_utterance_safe")
INITIAL_WEIGHTS_PATH = "/content/rawnet_model_initial_weights.pth"
RESULTS_ROOT = Path("/content/IRM_results")
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

SELECTED_SPLITS = [
    "split_00",
    "split_09",
    "split_14",
    "split_12",
]

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TARGET_SR = 16000
CLIP_SECONDS = 4
CLIP_LEN = TARGET_SR * CLIP_SECONDS  # 64000

# Split train sets have 4 source domains.
# DOMAIN_LABEL_BATCH_SIZE = 2 means:
# per domain: 2 real + 2 fake = 4 samples
# total batch: 4 domains * 4 samples = 16
DOMAIN_LABEL_BATCH_SIZE = 16
NUM_WORKERS = 4

EPOCHS = 10
LR = 1e-5
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 5.0

IRM_LAMBDA = 100.0
IRM_ANNEAL_EPOCHS = 2

EVAL_BATCH_SIZE = 16


# ============================================================
# Reproducibility
# ============================================================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def seed_worker(worker_id):
    worker_seed = (torch.initial_seed() + worker_id) % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


seed_everything(SEED)


# ============================================================
# Validation helpers
# ============================================================

def require_file(path, description):
    if not os.path.isfile(path):
        raise FileNotFoundError(f"{description} not found: {path}")


def validate_split_dataframe(df, csv_path, split_name, require_domain=False):
    required_cols = {"audio_path", "label"}

    if require_domain:
        required_cols.add("paired_domain")

    missing = required_cols - set(df.columns)

    if missing:
        raise ValueError(f"Missing columns in {csv_path}: {sorted(missing)}")

    if len(df) == 0:
        raise ValueError(f"{split_name} split is empty: {csv_path}")

    invalid_labels = sorted(set(df["label"].dropna().unique()) - {0, 1})

    if invalid_labels:
        raise ValueError(
            f"{split_name} split has labels outside {{0, 1}}: {invalid_labels}"
        )

    if df["label"].nunique(dropna=True) != 2:
        raise ValueError(
            f"{split_name} split must contain both labels 0 and 1, "
            f"but found: {sorted(df['label'].dropna().unique())}"
        )


def validate_no_missing_paths(df, csv_path, split_name, out_dir):
    exists_mask = df["audio_path"].apply(
        lambda p: isinstance(p, str) and os.path.isfile(p)
    )

    missing_df = df.loc[~exists_mask].copy()

    if len(missing_df) == 0:
        return df.reset_index(drop=True)

    report_path = os.path.join(out_dir, f"{split_name}_missing_audio_paths.csv")
    missing_df.to_csv(report_path, index=False)

    raise FileNotFoundError(
        f"{split_name} split has {len(missing_df)} missing audio files. "
        f"Report saved to: {report_path}"
    )


def validate_loader_not_empty(loader, split_name):
    if len(loader.dataset) == 0:
        raise ValueError(f"{split_name} DataLoader has an empty dataset")


def discover_split_dirs(splits_root, selected_splits):
    split_dirs = [splits_root / split_name for split_name in selected_splits]

    missing_dirs = [str(p) for p in split_dirs if not p.is_dir()]
    if missing_dirs:
        raise FileNotFoundError(f"Selected split directories not found: {missing_dirs}")

    for split_dir in split_dirs:
        require_file(split_dir / "main_train.csv", f"{split_dir.name} train CSV")
        require_file(split_dir / "main_val.csv", f"{split_dir.name} val CSV")
        require_file(split_dir / "main_test.csv", f"{split_dir.name} test CSV")

    return split_dirs


# ============================================================
# Dataset
# ============================================================

class AudioCSVDataset(Dataset):
    def __init__(
        self,
        csv_path,
        target_sr=16000,
        clip_len=64000,
        train=True,
        split_name="split",
        require_domain=False,
        out_dir=None,
    ):
        require_file(csv_path, f"{split_name} CSV")

        self.df = pd.read_csv(csv_path)
        self.target_sr = target_sr
        self.clip_len = clip_len
        self.train = train
        self.split_name = split_name
        self.out_dir = out_dir

        validate_split_dataframe(
            self.df,
            csv_path,
            split_name,
            require_domain=require_domain,
        )

        before = len(self.df)
        self.df = validate_no_missing_paths(self.df, csv_path, split_name, out_dir)
        after = len(self.df)

        print(f"\nLoaded: {csv_path}")
        print(f"Rows before path check: {before}")
        print(f"Rows after path check:  {after}")
        print("Label counts:")
        print(self.df["label"].value_counts().sort_index())

        if "paired_domain" in self.df.columns:
            print("Paired domain counts:")
            print(self.df.groupby(["paired_domain", "label"]).size())

    def __len__(self):
        return len(self.df)

    def _load_audio(self, path):
        try:
            wav, sr = torchaudio.load(path)
        except Exception as exc:
            raise RuntimeError(f"Failed to load audio file: {path}") from exc

        if wav.numel() == 0:
            raise RuntimeError(f"Audio file is empty: {path}")

        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)

        if sr != self.target_sr:
            wav = torchaudio.functional.resample(wav, sr, self.target_sr)

        wav = wav.squeeze(0)

        if not torch.isfinite(wav).all():
            raise RuntimeError(f"Audio contains NaN or Inf values: {path}")

        max_val = wav.abs().max()
        if max_val > 0:
            wav = wav / max_val

        return wav

    def _crop_or_pad(self, wav):
        T = wav.numel()

        if T == self.clip_len:
            return wav

        if T > self.clip_len:
            if self.train:
                start = random.randint(0, T - self.clip_len)
            else:
                start = (T - self.clip_len) // 2

            return wav[start:start + self.clip_len]

        pad_len = self.clip_len - T
        return torch.nn.functional.pad(wav, (0, pad_len))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        wav = self._load_audio(row["audio_path"])
        wav = self._crop_or_pad(wav)

        label = torch.tensor(int(row["label"]), dtype=torch.long)

        item = {
            "waveform": wav,
            "label": label,
            "audio_path": row["audio_path"],
        }

        if "paired_domain" in self.df.columns:
            item["paired_domain"] = row["paired_domain"]

        if "spoof_generator" in self.df.columns:
            item["spoof_generator"] = row["spoof_generator"]

        if "utterance_id" in self.df.columns:
            item["utterance_id"] = row["utterance_id"]

        return item


def collate_fn(batch):
    if len(batch) == 0:
        raise ValueError("Received an empty batch")

    waveforms = torch.stack([x["waveform"] for x in batch], dim=0)
    labels = torch.stack([x["label"] for x in batch], dim=0)

    out = {
        "waveform": waveforms,
        "label": labels,
        "audio_path": [x["audio_path"] for x in batch],
    }

    for key in ["paired_domain", "spoof_generator", "utterance_id"]:
        if key in batch[0]:
            out[key] = [x[key] for x in batch]

    return out


# ============================================================
# Domain-balanced batch sampler for IRM
# ============================================================

class DomainLabelBalancedBatchSampler(Sampler):
    """
    Creates batches that contain all source domains.

    For each domain, each batch contains:
        DOMAIN_LABEL_BATCH_SIZE real samples
        DOMAIN_LABEL_BATCH_SIZE fake samples

    So with 4 domains and DOMAIN_LABEL_BATCH_SIZE = 2:
        batch size = 4 * 2 * 2 = 16
    """

    def __init__(self, df, domain_col="paired_domain", label_col="label",
                 per_label_per_domain=2, seed=42):
        self.df = df.reset_index(drop=True)
        self.domain_col = domain_col
        self.label_col = label_col
        self.per_label_per_domain = per_label_per_domain
        self.seed = seed
        self.epoch = 0

        self.domains = sorted(self.df[domain_col].unique().tolist())

        self.indices = {}

        for domain in self.domains:
            for label in [0, 1]:
                idxs = self.df.index[
                    (self.df[domain_col] == domain) &
                    (self.df[label_col] == label)
                ].tolist()

                if len(idxs) < self.per_label_per_domain:
                    raise ValueError(
                        f"Not enough samples for domain={domain}, label={label}. "
                        f"Found {len(idxs)}, need at least {self.per_label_per_domain}."
                    )

                self.indices[(domain, label)] = idxs

        self.num_batches = min(
            len(v) // self.per_label_per_domain
            for v in self.indices.values()
        )

        self.batch_size = (
            len(self.domains) * 2 * self.per_label_per_domain
        )

        print("\nIRM batch sampler:")
        print("Source domains:", self.domains)
        print("Samples per label per domain per batch:", self.per_label_per_domain)
        print("Effective batch size:", self.batch_size)
        print("Batches per epoch:", self.num_batches)

    def __len__(self):
        return self.num_batches

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __iter__(self):
        rng = random.Random(self.seed + self.epoch)

        shuffled = {}

        for key, idxs in self.indices.items():
            idxs_copy = idxs.copy()
            rng.shuffle(idxs_copy)
            shuffled[key] = idxs_copy

        pointers = {key: 0 for key in shuffled.keys()}

        for _ in range(self.num_batches):
            batch = []

            for domain in self.domains:
                for label in [0, 1]:
                    key = (domain, label)
                    start = pointers[key]
                    end = start + self.per_label_per_domain

                    batch.extend(shuffled[key][start:end])
                    pointers[key] = end

            rng.shuffle(batch)
            yield batch


def make_loaders(split_dir, out_dir):
    train_csv = split_dir / "main_train.csv"
    val_csv = split_dir / "main_val.csv"
    test_csv = split_dir / "main_test.csv"

    train_ds = AudioCSVDataset(
        train_csv,
        TARGET_SR,
        CLIP_LEN,
        train=True,
        split_name=f"{split_dir.name}_train",
        require_domain=True,
        out_dir=out_dir,
    )

    val_ds = AudioCSVDataset(
        val_csv,
        TARGET_SR,
        CLIP_LEN,
        train=False,
        split_name=f"{split_dir.name}_val",
        require_domain=False,
        out_dir=out_dir,
    )

    test_ds = AudioCSVDataset(
        test_csv,
        TARGET_SR,
        CLIP_LEN,
        train=False,
        split_name=f"{split_dir.name}_test",
        require_domain=False,
        out_dir=out_dir,
    )

    irm_batch_sampler = DomainLabelBalancedBatchSampler(
        train_ds.df,
        domain_col="paired_domain",
        label_col="label",
        per_label_per_domain=DOMAIN_LABEL_BATCH_SIZE,
        seed=SEED,
    )

    train_loader = DataLoader(
        train_ds,
        batch_sampler=irm_batch_sampler,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
        collate_fn=collate_fn,
        worker_init_fn=seed_worker,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
        collate_fn=collate_fn,
        worker_init_fn=seed_worker,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
        collate_fn=collate_fn,
        worker_init_fn=seed_worker,
    )

    validate_loader_not_empty(train_loader, f"{split_dir.name} train")
    validate_loader_not_empty(val_loader, f"{split_dir.name} val")
    validate_loader_not_empty(test_loader, f"{split_dir.name} test")

    return train_loader, val_loader, test_loader, irm_batch_sampler


# ============================================================
# Model setup
# ============================================================

if "model" not in globals():
    raise NameError(
        "Expected a pre-constructed variable named `model`, but none was found. "
        "Create/load your RawNet model object before running this IRM cell."
    )

require_file(INITIAL_WEIGHTS_PATH, "Initial model weights")


def extract_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in ["model_state_dict", "state_dict", "model"]:
            if key in checkpoint and isinstance(checkpoint[key], dict):
                return checkpoint[key]

        if checkpoint and all(torch.is_tensor(v) for v in checkpoint.values()):
            return checkpoint

    raise ValueError(
        "Could not find a model state_dict in the initial weights file. "
        "Expected either a raw state_dict or a dict with model_state_dict/state_dict/model."
    )


def strip_module_prefix(state_dict):
    if not any(k.startswith("module.") for k in state_dict.keys()):
        return state_dict

    return {
        k.replace("module.", "", 1): v
        for k, v in state_dict.items()
    }


initial_checkpoint = torch.load(INITIAL_WEIGHTS_PATH, map_location="cpu")
initial_state_dict = strip_module_prefix(extract_state_dict(initial_checkpoint))

model = model.to(DEVICE)
model.float()

criterion = nn.CrossEntropyLoss(reduction="mean")


def reset_model_to_initial_weights(model):
    model.load_state_dict(initial_state_dict, strict=True)
    model.to(DEVICE)
    model.float()


def forward_model(model, waveforms):
    """
    waveforms: [B, 64000]
    logits:    [B, 2]
    """
    if waveforms.ndim != 2 or waveforms.shape[1] != CLIP_LEN:
        raise ValueError(f"Expected waveforms shape [B, {CLIP_LEN}], got {waveforms.shape}")

    logits = model(waveforms)

    if logits.ndim != 2 or logits.shape[1] != 2:
        raise ValueError(f"Expected logits shape [B, 2], got {logits.shape}")

    if not torch.isfinite(logits).all():
        raise RuntimeError("Model produced NaN or Inf logits")

    return logits


def cross_entropy_from_scaled_logits(logits, labels, scale):
    """
    IRM uses a dummy scale parameter.

    We do not modify the model.
    We multiply logits by scale only inside the loss.
    """
    return criterion(logits * scale, labels)


def irm_penalty(logits, labels):
    """
    IRMv1 penalty:
    gradient of risk with respect to a dummy classifier scale
    should be small and invariant across environments.
    """
    scale = torch.tensor(1.0, device=logits.device, requires_grad=True)

    loss = cross_entropy_from_scaled_logits(logits, labels, scale)

    grad = torch.autograd.grad(
        loss,
        [scale],
        create_graph=True,
    )[0]

    return torch.sum(grad ** 2)


def compute_irm_objective(logits, labels, domains, penalty_weight):
    unique_domains = sorted(set(domains))

    risks = []
    penalties = []

    for domain in unique_domains:
        mask_list = [d == domain for d in domains]
        mask = torch.tensor(mask_list, device=logits.device, dtype=torch.bool)

        domain_logits = logits[mask]
        domain_labels = labels[mask]

        if domain_logits.shape[0] == 0:
            continue

        risk = criterion(domain_logits, domain_labels)
        penalty = irm_penalty(domain_logits, domain_labels)

        risks.append(risk)
        penalties.append(penalty)

    if len(risks) == 0:
        raise RuntimeError("IRM batch contained no valid domains")

    mean_risk = torch.stack(risks).mean()
    mean_penalty = torch.stack(penalties).mean()

    objective = mean_risk + penalty_weight * mean_penalty

    return objective, mean_risk, mean_penalty


def compute_loss_and_scores(logits, labels):
    loss = criterion(logits, labels)
    scores = torch.softmax(logits, dim=1)[:, 1]
    return loss, scores


# ============================================================
# Metrics
# ============================================================

def compute_accuracy(labels, scores, threshold=0.5):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores).astype(float)

    if len(labels) == 0:
        return float("nan")

    preds = (scores >= threshold).astype(int)
    return float((preds == labels).mean())


def compute_eer(labels, scores):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores).astype(float)

    if len(labels) == 0:
        return float("nan")

    finite_mask = np.isfinite(scores)
    labels = labels[finite_mask]
    scores = scores[finite_mask]

    P = np.sum(labels == 1)
    N = np.sum(labels == 0)

    if P == 0 or N == 0:
        return float("nan")

    thresholds = np.r_[np.inf, np.unique(scores)[::-1], -np.inf]

    fars = []
    frrs = []

    for thr in thresholds:
        preds = (scores >= thr).astype(int)

        fp = np.sum((preds == 1) & (labels == 0))
        fn = np.sum((preds == 0) & (labels == 1))

        far = fp / N
        frr = fn / P

        fars.append(far)
        frrs.append(frr)

    fars = np.asarray(fars)
    frrs = np.asarray(frrs)

    idx = np.argmin(np.abs(fars - frrs))
    eer = (fars[idx] + frrs[idx]) / 2.0

    return float(eer)


@torch.no_grad()
def evaluate(model, loader, split_name="val"):
    model.eval()

    all_labels = []
    all_scores = []
    all_domains = []
    all_paths = []

    total_loss = 0.0
    total_n = 0

    for batch in loader:
        waveforms = batch["waveform"].to(DEVICE, non_blocking=True).float()
        labels = batch["label"].to(DEVICE, non_blocking=True).long()

        logits = forward_model(model, waveforms)
        loss, scores = compute_loss_and_scores(logits, labels)

        if not torch.isfinite(loss):
            raise RuntimeError(f"{split_name} evaluation produced invalid loss: {loss.item()}")

        bs = labels.shape[0]

        total_loss += loss.item() * bs
        total_n += bs

        all_labels.extend(labels.detach().cpu().numpy().tolist())
        all_scores.extend(scores.detach().cpu().numpy().tolist())
        all_paths.extend(batch["audio_path"])

        if "paired_domain" in batch:
            all_domains.extend(batch["paired_domain"])
        else:
            all_domains.extend(["unknown"] * bs)

    if total_n == 0:
        raise ValueError(f"{split_name} evaluation received zero samples")

    avg_loss = total_loss / total_n
    acc = compute_accuracy(all_labels, all_scores)
    eer = compute_eer(all_labels, all_scores)

    result = {
        "split": split_name,
        "loss": avg_loss,
        "accuracy": acc,
        "eer": eer,
        "num_samples": total_n,
    }

    pred_df = pd.DataFrame({
        "audio_path": all_paths,
        "label": all_labels,
        "score": all_scores,
        "paired_domain": all_domains,
    })

    per_domain_rows = []

    for domain, g in pred_df.groupby("paired_domain"):
        per_domain_rows.append({
            "split": split_name,
            "paired_domain": domain,
            "num_samples": len(g),
            "accuracy": compute_accuracy(g["label"].values, g["score"].values),
            "eer": compute_eer(g["label"].values, g["score"].values),
        })

    per_domain_df = pd.DataFrame(per_domain_rows)

    return result, per_domain_df, pred_df


# ============================================================
# Training
# ============================================================

def run_one_split(split_dir):
    split_name = split_dir.name
    out_dir = RESULTS_ROOT / split_name
    out_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 80)
    print(f"Starting IRM run: {split_name}")
    print(f"Outputs: {out_dir}")
    print("=" * 80)

    seed_everything(SEED)
    reset_model_to_initial_weights(model)

    train_loader, val_loader, test_loader, irm_batch_sampler = make_loaders(split_dir, out_dir)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    best_val_eer = float("inf")
    best_val_loss = float("inf")
    best_epoch = None
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        irm_batch_sampler.set_epoch(epoch)

        if epoch <= IRM_ANNEAL_EPOCHS:
            penalty_weight = 0.0
        else:
            penalty_weight = IRM_LAMBDA

        running_objective = 0.0
        running_risk = 0.0
        running_penalty = 0.0
        running_n = 0
        skipped_batches = 0

        train_pbar = tqdm(
            train_loader,
            desc=f"{split_name} | IRM Epoch {epoch:02d}/{EPOCHS}",
            leave=True,
            dynamic_ncols=True,
        )

        for batch in train_pbar:
            waveforms = batch["waveform"].to(DEVICE, non_blocking=True).float()
            labels = batch["label"].to(DEVICE, non_blocking=True).long()
            domains = batch["paired_domain"]

            optimizer.zero_grad(set_to_none=True)

            logits = forward_model(model, waveforms)

            objective, mean_risk, mean_penalty = compute_irm_objective(
                logits,
                labels,
                domains,
                penalty_weight=penalty_weight,
            )

            if not torch.isfinite(objective):
                skipped_batches += 1
                warnings.warn(f"Skipping batch due to invalid objective: {objective.item()}")
                continue

            objective.backward()

            if GRAD_CLIP is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

            optimizer.step()

            bs = labels.shape[0]
            running_objective += objective.item() * bs
            running_risk += mean_risk.item() * bs
            running_penalty += mean_penalty.item() * bs
            running_n += bs

            train_pbar.set_postfix({
                "obj": f"{running_objective / max(running_n, 1):.4f}",
                "risk": f"{running_risk / max(running_n, 1):.4f}",
                "pen": f"{running_penalty / max(running_n, 1):.4f}",
                "lambda": penalty_weight,
                "skipped": skipped_batches,
            })

        if running_n == 0:
            raise RuntimeError(
                f"{split_name} epoch {epoch} completed with zero usable training samples. "
                f"Skipped batches: {skipped_batches}"
            )

        train_objective = running_objective / running_n
        train_risk = running_risk / running_n
        train_penalty = running_penalty / running_n

        val_result, val_per_domain_df, val_pred_df = evaluate(
            model,
            val_loader,
            split_name="val",
        )

        print(
            f"\n{split_name} | IRM Epoch {epoch:02d} done | "
            f"Train obj: {train_objective:.4f} | "
            f"Train risk: {train_risk:.4f} | "
            f"Train penalty: {train_penalty:.6f} | "
            f"Penalty weight: {penalty_weight} | "
            f"Val loss: {val_result['loss']:.4f} | "
            f"Val acc: {val_result['accuracy']:.4f} | "
            f"Val EER: {val_result['eer']:.4f} | "
            f"Skipped batches: {skipped_batches}\n"
        )

        epoch_record = {
            "split_dir": str(split_dir),
            "run": split_name,
            "epoch": epoch,
            "train_objective": train_objective,
            "train_risk": train_risk,
            "train_penalty": train_penalty,
            "penalty_weight": penalty_weight,
            "val_loss": val_result["loss"],
            "val_accuracy": val_result["accuracy"],
            "val_eer": val_result["eer"],
            "val_num_samples": val_result["num_samples"],
            "skipped_batches": skipped_batches,
        }

        history.append(epoch_record)

        val_eer = val_result["eer"]
        val_loss = val_result["loss"]

        if math.isfinite(val_eer):
            is_new_best = val_eer < best_val_eer
        else:
            is_new_best = not math.isfinite(best_val_eer) and val_loss < best_val_loss

        if is_new_best:
            best_val_eer = val_eer
            best_val_loss = val_loss
            best_epoch = epoch

            ckpt_path = out_dir / "best_irm.pt"

            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_eer": best_val_eer,
                "best_val_loss": best_val_loss,
                "config": {
                    "split_dir": str(split_dir),
                    "initial_weights_path": INITIAL_WEIGHTS_PATH,
                    "target_sr": TARGET_SR,
                    "clip_seconds": CLIP_SECONDS,
                    "clip_len": CLIP_LEN,
                    "domain_label_batch_size": DOMAIN_LABEL_BATCH_SIZE,
                    "effective_batch_size": irm_batch_sampler.batch_size,
                    "eval_batch_size": EVAL_BATCH_SIZE,
                    "lr": LR,
                    "weight_decay": WEIGHT_DECAY,
                    "epochs": EPOCHS,
                    "irm_lambda": IRM_LAMBDA,
                    "irm_anneal_epochs": IRM_ANNEAL_EPOCHS,
                    "seed": SEED,
                }
            }, ckpt_path)

            val_pred_df.to_csv(out_dir / "best_val_predictions.csv", index=False)
            val_per_domain_df.to_csv(out_dir / "best_val_per_domain.csv", index=False)

            print(f"Saved new best checkpoint: {ckpt_path}")
            print(f"Best val EER: {best_val_eer:.4f}")

    pd.DataFrame(history).to_csv(out_dir / "training_history.csv", index=False)

    best_ckpt_path = out_dir / "best_irm.pt"

    if not best_ckpt_path.is_file():
        raise RuntimeError(
            f"No best IRM checkpoint was saved for {split_name}. Check validation metrics."
        )

    best_ckpt = torch.load(best_ckpt_path, map_location=DEVICE)

    model.load_state_dict(best_ckpt["model_state_dict"])
    model.to(DEVICE)
    model.float()

    test_result, test_per_domain_df, test_pred_df = evaluate(
        model,
        test_loader,
        split_name="test",
    )

    test_result["run"] = split_name
    test_result["split_dir"] = str(split_dir)
    test_result["best_epoch"] = best_epoch
    test_result["best_val_eer"] = best_val_eer
    test_result["best_val_loss"] = best_val_loss
    test_result["irm_lambda"] = IRM_LAMBDA
    test_result["irm_anneal_epochs"] = IRM_ANNEAL_EPOCHS
    test_result["domain_label_batch_size"] = DOMAIN_LABEL_BATCH_SIZE
    test_result["effective_batch_size"] = irm_batch_sampler.batch_size
    test_result["initial_weights_path"] = INITIAL_WEIGHTS_PATH

    print(f"\n{split_name} | Final IRM Test Result:")
    print(test_result)

    print(f"\n{split_name} | IRM Test Per Domain:")
    print(test_per_domain_df)

    test_pred_df.to_csv(out_dir / "test_predictions.csv", index=False)
    test_per_domain_df.to_csv(out_dir / "test_per_domain.csv", index=False)

    with open(out_dir / "test_result.json", "w") as f:
        json.dump(test_result, f, indent=2)

    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    print(f"\nSaved IRM outputs to: {out_dir}")
    return test_result


# ============================================================
# Run selected splits
# ============================================================

split_dirs = discover_split_dirs(SPLITS_ROOT, SELECTED_SPLITS)
print(f"Found {len(split_dirs)} selected split directories under {SPLITS_ROOT}")
print(f"Selected splits: {', '.join(p.name for p in split_dirs)}")
print(f"IRM lambda: {IRM_LAMBDA}")
print(f"IRM anneal epochs: {IRM_ANNEAL_EPOCHS}")
print(f"Domain-label batch size: {DOMAIN_LABEL_BATCH_SIZE}")
print(f"Initial weights: {INITIAL_WEIGHTS_PATH}")
print(f"Results root: {RESULTS_ROOT}")

all_results = []

for split_dir in split_dirs:
    split_result = run_one_split(split_dir)
    all_results.append(split_result)

all_results_df = pd.DataFrame(all_results)
all_results_path = RESULTS_ROOT / "all_splits_test_results.csv"
all_results_df.to_csv(all_results_path, index=False)

summary = {
    "num_splits": len(all_results_df),
    "selected_splits": SELECTED_SPLITS,
    "mean_test_accuracy": float(all_results_df["accuracy"].mean()),
    "std_test_accuracy": float(all_results_df["accuracy"].std(ddof=0)),
    "mean_test_eer": float(all_results_df["eer"].mean()),
    "std_test_eer": float(all_results_df["eer"].std(ddof=0)),
    "irm_lambda": IRM_LAMBDA,
    "irm_anneal_epochs": IRM_ANNEAL_EPOCHS,
    "domain_label_batch_size": DOMAIN_LABEL_BATCH_SIZE,
    "results_csv": str(all_results_path),
}

with open(RESULTS_ROOT / "all_splits_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nSelected IRM splits complete.")
print(all_results_df)
print("\nAggregate summary:")
print(summary)
print("Saved aggregate results:", all_results_path)


Found 4 selected split directories under /content/dg_splits_utterance_safe
Selected splits: split_00, split_09, split_14, split_12
IRM lambda: 100.0
IRM anneal epochs: 2
Domain-label batch size: 16
Initial weights: /content/rawnet_model_initial_weights.pth
Results root: /content/IRM_results

Starting IRM run: split_00
Outputs: /content/IRM_results/split_00

Loaded: /content/dg_splits_utterance_safe/split_00/main_train.csv
Rows before path check: 6268
Rows after path check:  6268
Label counts:
label
0    3134
1    3134
Name: count, dtype: int64
Paired domain counts:
paired_domain                                    label
tts_models_fr_css10_vits                         0        781
                                                 1        781
tts_models_fr_mai_tacotron2-DDC                  0        786
                                                 1        786
tts_models_multilingual_multi-dataset_bark       0        794
                                                 1        794
t

split_00 | IRM Epoch 01/10:   0%|          | 0/48 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x789c411dd260>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x789c411dd260>^
^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^    self._shutdown_workers()^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^    ^if w.is_alive():^
 ^ ^ ^ ^ ^  ^^^


split_00 | IRM Epoch 01 done | Train obj: 0.5133 | Train risk: 0.5133 | Train penalty: 0.009703 | Penalty weight: 0.0 | Val loss: 2.5160 | Val acc: 0.5000 | Val EER: 0.3838 | Skipped batches: 0

Saved new best checkpoint: /content/IRM_results/split_00/best_irm.pt
Best val EER: 0.3838


split_00 | IRM Epoch 02/10:   0%|          | 0/48 [00:00<?, ?it/s]

...

split_12 | IRM Epoch 09/10:   0%|          | 0/47 [00:00<?, ?it/s]

Exception ignored in: Exception ignored in: Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x789c411dd260><function _MultiProcessingDataLoaderIter.__del__ at 0x789c411dd260>
Traceback (most recent call last):
<function _MultiProcessingDataLoaderIter.__del__ at 0x789c411dd260>
<function _MultiProcessingDataLoaderIter.__del__ at 0x789c411dd260>Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    

self._shutdown_workers()Traceback (most recent call last):
Traceback (most recent call last):

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
            self._shutdown_workers()self._shutdow


split_12 | IRM Epoch 09 done | Train obj: 0.6611 | Train risk: 0.6176 | Train penalty: 0.000435 | Penalty weight: 100.0 | Val loss: 0.5966 | Val acc: 0.5425 | Val EER: 0.1242 | Skipped batches: 0

Saved new best checkpoint: /content/IRM_results/split_12/best_irm.pt
Best val EER: 0.1242


split_12 | IRM Epoch 10/10:   0%|          | 0/47 [00:00<?, ?it/s]


split_12 | IRM Epoch 10 done | Train obj: 0.6545 | Train risk: 0.5922 | Train penalty: 0.000624 | Penalty weight: 100.0 | Val loss: 0.5874 | Val acc: 0.5108 | Val EER: 0.1115 | Skipped batches: 0

Saved new best checkpoint: /content/IRM_results/split_12/best_irm.pt
Best val EER: 0.1115

split_12 | Final IRM Test Result:
{'split': 'test', 'loss': 0.6851438481807709, 'accuracy': 0.5, 'eer': 0.341, 'num_samples': 4000, 'run': 'split_12', 'split_dir': '/content/dg_splits_utterance_safe/split_12', 'best_epoch': 10, 'best_val_eer': 0.11153358681875793, 'best_val_loss': 0.5874424839276626, 'irm_lambda': 100.0, 'irm_anneal_epochs': 2, 'domain_label_batch_size': 16, 'effective_batch_size': 128, 'initial_weights_path': '/content/rawnet_model_initial_weights.pth'}

split_12 | IRM Test Per Domain:
  split                               paired_domain  num_samples  accuracy  \
0  test             tts_models_fr_mai_tacotron2-DDC         2000     0.503   
1  test  tts_models_multilingual_multi-dataset

### Export IRM results

In [ ]:
prepare_and_download('/content/IRM_results', '/content/IRM_results_clean', 'IRM_results_clean.tar.gz')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Archived /content/IRM_results (sans weights) and started download.


## GroupDRO

In [ ]:
import os
import random
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler

import torchaudio
from tqdm.auto import tqdm


# ============================================================
# Config
# ============================================================

SPLITS_ROOT = Path("/content/dg_splits_utterance_safe")
INITIAL_WEIGHTS_PATH = "/content/rawnet_model_initial_weights.pth"
RESULTS_ROOT = Path("/content/GROUP_DRO_results")
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

SELECTED_SPLITS = [
    "split_00",
    "split_09",
    "split_14",
    "split_12",
]

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TARGET_SR = 16000
CLIP_SECONDS = 4
CLIP_LEN = TARGET_SR * CLIP_SECONDS  # 64000

# Same batch design as IRM:
# per domain: DOMAIN_LABEL_BATCH_SIZE real + DOMAIN_LABEL_BATCH_SIZE fake
# effective batch size = 4 domains * 2 labels * DOMAIN_LABEL_BATCH_SIZE
DOMAIN_LABEL_BATCH_SIZE = 16
NUM_WORKERS = 4

EPOCHS = 10
LR = 1e-5
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 5.0

EVAL_BATCH_SIZE = 16

# Group DRO hyperparameter.
# Larger eta makes q focus more aggressively on high-loss domains.
GROUP_DRO_ETA = 0.01


# ============================================================
# Reproducibility
# ============================================================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def seed_worker(worker_id):
    worker_seed = (torch.initial_seed() + worker_id) % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


seed_everything(SEED)


# ============================================================
# Validation helpers
# ============================================================

def require_file(path, description):
    if not os.path.isfile(path):
        raise FileNotFoundError(f"{description} not found: {path}")


def validate_split_dataframe(df, csv_path, split_name, require_domain=False):
    required_cols = {"audio_path", "label"}

    if require_domain:
        required_cols.add("paired_domain")

    missing = required_cols - set(df.columns)

    if missing:
        raise ValueError(f"Missing columns in {csv_path}: {sorted(missing)}")

    if len(df) == 0:
        raise ValueError(f"{split_name} split is empty: {csv_path}")

    invalid_labels = sorted(set(df["label"].dropna().unique()) - {0, 1})

    if invalid_labels:
        raise ValueError(
            f"{split_name} split has labels outside {{0, 1}}: {invalid_labels}"
        )

    if df["label"].nunique(dropna=True) != 2:
        raise ValueError(
            f"{split_name} split must contain both labels 0 and 1, "
            f"but found: {sorted(df['label'].dropna().unique())}"
        )


def validate_no_missing_paths(df, csv_path, split_name, out_dir):
    exists_mask = df["audio_path"].apply(
        lambda p: isinstance(p, str) and os.path.isfile(p)
    )

    missing_df = df.loc[~exists_mask].copy()

    if len(missing_df) == 0:
        return df.reset_index(drop=True)

    report_path = os.path.join(out_dir, f"{split_name}_missing_audio_paths.csv")
    missing_df.to_csv(report_path, index=False)

    raise FileNotFoundError(
        f"{split_name} split has {len(missing_df)} missing audio files. "
        f"Report saved to: {report_path}"
    )


def validate_loader_not_empty(loader, split_name):
    if len(loader.dataset) == 0:
        raise ValueError(f"{split_name} DataLoader has an empty dataset")


def discover_split_dirs(splits_root, selected_splits):
    split_dirs = [splits_root / split_name for split_name in selected_splits]

    missing_dirs = [str(p) for p in split_dirs if not p.is_dir()]
    if missing_dirs:
        raise FileNotFoundError(f"Selected split directories not found: {missing_dirs}")

    for split_dir in split_dirs:
        require_file(split_dir / "main_train.csv", f"{split_dir.name} train CSV")
        require_file(split_dir / "main_val.csv", f"{split_dir.name} val CSV")
        require_file(split_dir / "main_test.csv", f"{split_dir.name} test CSV")

    return split_dirs


# ============================================================
# Dataset
# ============================================================

class AudioCSVDataset(Dataset):
    def __init__(
        self,
        csv_path,
        target_sr=16000,
        clip_len=64000,
        train=True,
        split_name="split",
        require_domain=False,
        out_dir=None,
    ):
        require_file(csv_path, f"{split_name} CSV")

        self.df = pd.read_csv(csv_path)
        self.target_sr = target_sr
        self.clip_len = clip_len
        self.train = train
        self.split_name = split_name
        self.out_dir = out_dir

        validate_split_dataframe(
            self.df,
            csv_path,
            split_name,
            require_domain=require_domain,
        )

        before = len(self.df)
        self.df = validate_no_missing_paths(self.df, csv_path, split_name, out_dir)
        after = len(self.df)

        print(f"\nLoaded: {csv_path}")
        print(f"Rows before path check: {before}")
        print(f"Rows after path check:  {after}")
        print("Label counts:")
        print(self.df["label"].value_counts().sort_index())

        if "paired_domain" in self.df.columns:
            print("Paired domain counts:")
            print(self.df.groupby(["paired_domain", "label"]).size())

    def __len__(self):
        return len(self.df)

    def _load_audio(self, path):
        try:
            wav, sr = torchaudio.load(path)
        except Exception as exc:
            raise RuntimeError(f"Failed to load audio file: {path}") from exc

        if wav.numel() == 0:
            raise RuntimeError(f"Audio file is empty: {path}")

        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)

        if sr != self.target_sr:
            wav = torchaudio.functional.resample(wav, sr, self.target_sr)

        wav = wav.squeeze(0)

        if not torch.isfinite(wav).all():
            raise RuntimeError(f"Audio contains NaN or Inf values: {path}")

        max_val = wav.abs().max()
        if max_val > 0:
            wav = wav / max_val

        return wav

    def _crop_or_pad(self, wav):
        T = wav.numel()

        if T == self.clip_len:
            return wav

        if T > self.clip_len:
            if self.train:
                start = random.randint(0, T - self.clip_len)
            else:
                start = (T - self.clip_len) // 2

            return wav[start:start + self.clip_len]

        pad_len = self.clip_len - T
        return torch.nn.functional.pad(wav, (0, pad_len))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        wav = self._load_audio(row["audio_path"])
        wav = self._crop_or_pad(wav)

        label = torch.tensor(int(row["label"]), dtype=torch.long)

        item = {
            "waveform": wav,
            "label": label,
            "audio_path": row["audio_path"],
        }

        if "paired_domain" in self.df.columns:
            item["paired_domain"] = row["paired_domain"]

        if "spoof_generator" in self.df.columns:
            item["spoof_generator"] = row["spoof_generator"]

        if "utterance_id" in self.df.columns:
            item["utterance_id"] = row["utterance_id"]

        return item


def collate_fn(batch):
    if len(batch) == 0:
        raise ValueError("Received an empty batch")

    waveforms = torch.stack([x["waveform"] for x in batch], dim=0)
    labels = torch.stack([x["label"] for x in batch], dim=0)

    out = {
        "waveform": waveforms,
        "label": labels,
        "audio_path": [x["audio_path"] for x in batch],
    }

    for key in ["paired_domain", "spoof_generator", "utterance_id"]:
        if key in batch[0]:
            out[key] = [x[key] for x in batch]

    return out


# ============================================================
# Domain-label balanced batch sampler
# ============================================================

class DomainLabelBalancedBatchSampler(Sampler):
    """
    Creates batches that contain all source domains.

    For each domain, each batch contains:
        DOMAIN_LABEL_BATCH_SIZE real samples
        DOMAIN_LABEL_BATCH_SIZE fake samples

    With 4 domains and DOMAIN_LABEL_BATCH_SIZE = 2:
        batch size = 4 * 2 * 2 = 16
    """

    def __init__(
        self,
        df,
        domain_col="paired_domain",
        label_col="label",
        per_label_per_domain=2,
        seed=42,
    ):
        self.df = df.reset_index(drop=True)
        self.domain_col = domain_col
        self.label_col = label_col
        self.per_label_per_domain = per_label_per_domain
        self.seed = seed
        self.epoch = 0

        self.domains = sorted(self.df[domain_col].unique().tolist())

        self.indices = {}

        for domain in self.domains:
            for label in [0, 1]:
                idxs = self.df.index[
                    (self.df[domain_col] == domain) &
                    (self.df[label_col] == label)
                ].tolist()

                if len(idxs) < self.per_label_per_domain:
                    raise ValueError(
                        f"Not enough samples for domain={domain}, label={label}. "
                        f"Found {len(idxs)}, need at least {self.per_label_per_domain}."
                    )

                self.indices[(domain, label)] = idxs

        self.num_batches = min(
            len(v) // self.per_label_per_domain
            for v in self.indices.values()
        )

        self.batch_size = len(self.domains) * 2 * self.per_label_per_domain

        print("\nGroup DRO batch sampler:")
        print("Source domains:", self.domains)
        print("Samples per label per domain per batch:", self.per_label_per_domain)
        print("Effective batch size:", self.batch_size)
        print("Batches per epoch:", self.num_batches)

    def __len__(self):
        return self.num_batches

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __iter__(self):
        rng = random.Random(self.seed + self.epoch)

        shuffled = {}

        for key, idxs in self.indices.items():
            idxs_copy = idxs.copy()
            rng.shuffle(idxs_copy)
            shuffled[key] = idxs_copy

        pointers = {key: 0 for key in shuffled.keys()}

        for _ in range(self.num_batches):
            batch = []

            for domain in self.domains:
                for label in [0, 1]:
                    key = (domain, label)
                    start = pointers[key]
                    end = start + self.per_label_per_domain

                    batch.extend(shuffled[key][start:end])
                    pointers[key] = end

            rng.shuffle(batch)
            yield batch


def make_loaders(split_dir, out_dir):
    train_csv = split_dir / "main_train.csv"
    val_csv = split_dir / "main_val.csv"
    test_csv = split_dir / "main_test.csv"

    train_ds = AudioCSVDataset(
        train_csv,
        TARGET_SR,
        CLIP_LEN,
        train=True,
        split_name=f"{split_dir.name}_train",
        require_domain=True,
        out_dir=out_dir,
    )

    val_ds = AudioCSVDataset(
        val_csv,
        TARGET_SR,
        CLIP_LEN,
        train=False,
        split_name=f"{split_dir.name}_val",
        require_domain=False,
        out_dir=out_dir,
    )

    test_ds = AudioCSVDataset(
        test_csv,
        TARGET_SR,
        CLIP_LEN,
        train=False,
        split_name=f"{split_dir.name}_test",
        require_domain=False,
        out_dir=out_dir,
    )

    group_dro_batch_sampler = DomainLabelBalancedBatchSampler(
        train_ds.df,
        domain_col="paired_domain",
        label_col="label",
        per_label_per_domain=DOMAIN_LABEL_BATCH_SIZE,
        seed=SEED,
    )

    train_loader = DataLoader(
        train_ds,
        batch_sampler=group_dro_batch_sampler,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
        collate_fn=collate_fn,
        worker_init_fn=seed_worker,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
        collate_fn=collate_fn,
        worker_init_fn=seed_worker,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
        collate_fn=collate_fn,
        worker_init_fn=seed_worker,
    )

    validate_loader_not_empty(train_loader, f"{split_dir.name} train")
    validate_loader_not_empty(val_loader, f"{split_dir.name} val")
    validate_loader_not_empty(test_loader, f"{split_dir.name} test")

    return train_loader, val_loader, test_loader, group_dro_batch_sampler


# ============================================================
# Model setup
# ============================================================

if "model" not in globals():
    raise NameError(
        "Expected a pre-constructed variable named `model`, but none was found. "
        "Create/load your RawNet model object before running this Group DRO cell."
    )

require_file(INITIAL_WEIGHTS_PATH, "Initial model weights")


def extract_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in ["model_state_dict", "state_dict", "model"]:
            if key in checkpoint and isinstance(checkpoint[key], dict):
                return checkpoint[key]

        if checkpoint and all(torch.is_tensor(v) for v in checkpoint.values()):
            return checkpoint

    raise ValueError(
        "Could not find a model state_dict in the initial weights file. "
        "Expected either a raw state_dict or a dict with model_state_dict/state_dict/model."
    )


def strip_module_prefix(state_dict):
    if not any(k.startswith("module.") for k in state_dict.keys()):
        return state_dict

    return {
        k.replace("module.", "", 1): v
        for k, v in state_dict.items()
    }


initial_checkpoint = torch.load(INITIAL_WEIGHTS_PATH, map_location="cpu")
initial_state_dict = strip_module_prefix(extract_state_dict(initial_checkpoint))

model = model.to(DEVICE)
model.float()

criterion = nn.CrossEntropyLoss(reduction="mean")


def reset_model_to_initial_weights(model):
    model.load_state_dict(initial_state_dict, strict=True)
    model.to(DEVICE)
    model.float()


def forward_model(model, waveforms):
    """
    waveforms: [B, 64000]
    logits:    [B, 2]
    """
    if waveforms.ndim != 2 or waveforms.shape[1] != CLIP_LEN:
        raise ValueError(
            f"Expected waveforms shape [B, {CLIP_LEN}], got {waveforms.shape}"
        )

    logits = model(waveforms)

    if logits.ndim != 2 or logits.shape[1] != 2:
        raise ValueError(f"Expected logits shape [B, 2], got {logits.shape}")

    if not torch.isfinite(logits).all():
        raise RuntimeError("Model produced NaN or Inf logits")

    return logits


def compute_loss_and_scores(logits, labels):
    loss = criterion(logits, labels)
    scores = torch.softmax(logits, dim=1)[:, 1]
    return loss, scores


# ============================================================
# Group DRO objective
# ============================================================

class GroupDROState:
    """
    Maintains adversarial group weights q.

    q starts uniform over source domains.
    On each batch:
        q_g <- q_g * exp(eta * loss_g)
        q <- q / sum(q)

    The model then minimizes:
        sum_g q_g * loss_g

    This emphasizes groups/domains that currently have higher loss.
    """

    def __init__(self, domains, eta=0.01, device="cpu"):
        self.domains = sorted(domains)
        self.domain_to_idx = {d: i for i, d in enumerate(self.domains)}
        self.eta = eta
        self.device = device

        self.q = torch.ones(len(self.domains), device=device)
        self.q = self.q / self.q.sum()

    def reset(self):
        self.q = torch.ones(len(self.domains), device=self.device)
        self.q = self.q / self.q.sum()

    @torch.no_grad()
    def update_q(self, detached_group_losses):
        """
        detached_group_losses: tensor [num_domains]
        """
        self.q *= torch.exp(self.eta * detached_group_losses)
        self.q /= self.q.sum()

    def q_dict(self):
        return {
            domain: float(self.q[idx].detach().cpu())
            for domain, idx in self.domain_to_idx.items()
        }


def compute_group_dro_objective(logits, labels, domains, group_dro_state):
    group_losses = []

    for domain in group_dro_state.domains:
        mask_list = [d == domain for d in domains]
        mask = torch.tensor(mask_list, device=logits.device, dtype=torch.bool)

        domain_logits = logits[mask]
        domain_labels = labels[mask]

        if domain_logits.shape[0] == 0:
            raise RuntimeError(
                f"Group DRO batch is missing domain={domain}. "
                "This should not happen with the balanced batch sampler."
            )

        domain_loss = criterion(domain_logits, domain_labels)
        group_losses.append(domain_loss)

    group_losses = torch.stack(group_losses)  # [G]

    group_dro_state.update_q(group_losses.detach())

    objective = torch.sum(group_dro_state.q.detach() * group_losses)
    mean_loss = group_losses.mean()
    worst_group_loss = group_losses.max()

    return objective, mean_loss, worst_group_loss, group_losses


# ============================================================
# Metrics
# ============================================================

def compute_accuracy(labels, scores, threshold=0.5):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores).astype(float)

    if len(labels) == 0:
        return float("nan")

    preds = (scores >= threshold).astype(int)
    return float((preds == labels).mean())


def compute_eer(labels, scores):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores).astype(float)

    if len(labels) == 0:
        return float("nan")

    finite_mask = np.isfinite(scores)
    labels = labels[finite_mask]
    scores = scores[finite_mask]

    P = np.sum(labels == 1)
    N = np.sum(labels == 0)

    if P == 0 or N == 0:
        return float("nan")

    thresholds = np.r_[np.inf, np.unique(scores)[::-1], -np.inf]

    fars = []
    frrs = []

    for thr in thresholds:
        preds = (scores >= thr).astype(int)

        fp = np.sum((preds == 1) & (labels == 0))
        fn = np.sum((preds == 0) & (labels == 1))

        far = fp / N
        frr = fn / P

        fars.append(far)
        frrs.append(frr)

    fars = np.asarray(fars)
    frrs = np.asarray(frrs)

    idx = np.argmin(np.abs(fars - frrs))
    eer = (fars[idx] + frrs[idx]) / 2.0

    return float(eer)


@torch.no_grad()
def evaluate(model, loader, split_name="val"):
    model.eval()

    all_labels = []
    all_scores = []
    all_domains = []
    all_paths = []

    total_loss = 0.0
    total_n = 0

    for batch in loader:
        waveforms = batch["waveform"].to(DEVICE, non_blocking=True).float()
        labels = batch["label"].to(DEVICE, non_blocking=True).long()

        logits = forward_model(model, waveforms)
        loss, scores = compute_loss_and_scores(logits, labels)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"{split_name} evaluation produced invalid loss: {loss.item()}"
            )

        bs = labels.shape[0]

        total_loss += loss.item() * bs
        total_n += bs

        all_labels.extend(labels.detach().cpu().numpy().tolist())
        all_scores.extend(scores.detach().cpu().numpy().tolist())
        all_paths.extend(batch["audio_path"])

        if "paired_domain" in batch:
            all_domains.extend(batch["paired_domain"])
        else:
            all_domains.extend(["unknown"] * bs)

    if total_n == 0:
        raise ValueError(f"{split_name} evaluation received zero samples")

    avg_loss = total_loss / total_n
    acc = compute_accuracy(all_labels, all_scores)
    eer = compute_eer(all_labels, all_scores)

    result = {
        "split": split_name,
        "loss": avg_loss,
        "accuracy": acc,
        "eer": eer,
        "num_samples": total_n,
    }

    pred_df = pd.DataFrame({
        "audio_path": all_paths,
        "label": all_labels,
        "score": all_scores,
        "paired_domain": all_domains,
    })

    per_domain_rows = []

    for domain, g in pred_df.groupby("paired_domain"):
        per_domain_rows.append({
            "split": split_name,
            "paired_domain": domain,
            "num_samples": len(g),
            "accuracy": compute_accuracy(g["label"].values, g["score"].values),
            "eer": compute_eer(g["label"].values, g["score"].values),
        })

    per_domain_df = pd.DataFrame(per_domain_rows)

    return result, per_domain_df, pred_df


# ============================================================
# Training
# ============================================================

def run_one_split(split_dir):
    split_name = split_dir.name
    out_dir = RESULTS_ROOT / split_name
    out_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 80)
    print(f"Starting Group DRO run: {split_name}")
    print(f"Outputs: {out_dir}")
    print("=" * 80)

    seed_everything(SEED)
    reset_model_to_initial_weights(model)

    train_loader, val_loader, test_loader, group_dro_batch_sampler = make_loaders(
        split_dir,
        out_dir,
    )

    group_dro_state = GroupDROState(
        domains=group_dro_batch_sampler.domains,
        eta=GROUP_DRO_ETA,
        device=DEVICE,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    best_val_eer = float("inf")
    best_val_loss = float("inf")
    best_epoch = None
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        group_dro_batch_sampler.set_epoch(epoch)

        running_objective = 0.0
        running_mean_loss = 0.0
        running_worst_loss = 0.0
        running_n = 0
        skipped_batches = 0

        train_pbar = tqdm(
            train_loader,
            desc=f"{split_name} | GroupDRO Epoch {epoch:02d}/{EPOCHS}",
            leave=True,
            dynamic_ncols=True,
        )

        for batch in train_pbar:
            waveforms = batch["waveform"].to(DEVICE, non_blocking=True).float()
            labels = batch["label"].to(DEVICE, non_blocking=True).long()
            domains = batch["paired_domain"]

            optimizer.zero_grad(set_to_none=True)

            logits = forward_model(model, waveforms)

            objective, mean_loss, worst_group_loss, group_losses = compute_group_dro_objective(
                logits,
                labels,
                domains,
                group_dro_state,
            )

            if not torch.isfinite(objective):
                skipped_batches += 1
                warnings.warn(
                    f"Skipping batch due to invalid objective: {objective.item()}"
                )
                continue

            objective.backward()

            if GRAD_CLIP is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

            optimizer.step()

            bs = labels.shape[0]
            running_objective += objective.item() * bs
            running_mean_loss += mean_loss.item() * bs
            running_worst_loss += worst_group_loss.item() * bs
            running_n += bs

            train_pbar.set_postfix({
                "obj": f"{running_objective / max(running_n, 1):.4f}",
                "mean": f"{running_mean_loss / max(running_n, 1):.4f}",
                "worst": f"{running_worst_loss / max(running_n, 1):.4f}",
                "skipped": skipped_batches,
            })

        if running_n == 0:
            raise RuntimeError(
                f"{split_name} epoch {epoch} completed with zero usable training samples. "
                f"Skipped batches: {skipped_batches}"
            )

        train_objective = running_objective / running_n
        train_mean_loss = running_mean_loss / running_n
        train_worst_group_loss = running_worst_loss / running_n

        val_result, val_per_domain_df, val_pred_df = evaluate(
            model,
            val_loader,
            split_name="val",
        )

        current_q = group_dro_state.q_dict()

        print(
            f"\n{split_name} | Group DRO Epoch {epoch:02d} done | "
            f"Train obj: {train_objective:.4f} | "
            f"Train mean loss: {train_mean_loss:.4f} | "
            f"Train worst-group loss: {train_worst_group_loss:.4f} | "
            f"Val loss: {val_result['loss']:.4f} | "
            f"Val acc: {val_result['accuracy']:.4f} | "
            f"Val EER: {val_result['eer']:.4f} | "
            f"Skipped batches: {skipped_batches}\n"
        )

        print("Current Group DRO q weights:")
        print(current_q)

        epoch_record = {
            "split_dir": str(split_dir),
            "run": split_name,
            "epoch": epoch,
            "train_objective": train_objective,
            "train_mean_loss": train_mean_loss,
            "train_worst_group_loss": train_worst_group_loss,
            "val_loss": val_result["loss"],
            "val_accuracy": val_result["accuracy"],
            "val_eer": val_result["eer"],
            "val_num_samples": val_result["num_samples"],
            "skipped_batches": skipped_batches,
            "group_dro_eta": GROUP_DRO_ETA,
        }

        for domain, q_value in current_q.items():
            epoch_record[f"q_{domain}"] = q_value

        history.append(epoch_record)

        val_eer = val_result["eer"]
        val_loss = val_result["loss"]

        if math.isfinite(val_eer):
            is_new_best = val_eer < best_val_eer
        else:
            is_new_best = not math.isfinite(best_val_eer) and val_loss < best_val_loss

        if is_new_best:
            best_val_eer = val_eer
            best_val_loss = val_loss
            best_epoch = epoch

            ckpt_path = out_dir / "best_group_dro.pt"

            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_eer": best_val_eer,
                "best_val_loss": best_val_loss,
                "group_dro_q": group_dro_state.q.detach().cpu(),
                "group_dro_domains": group_dro_state.domains,
                "config": {
                    "split_dir": str(split_dir),
                    "initial_weights_path": INITIAL_WEIGHTS_PATH,
                    "target_sr": TARGET_SR,
                    "clip_seconds": CLIP_SECONDS,
                    "clip_len": CLIP_LEN,
                    "domain_label_batch_size": DOMAIN_LABEL_BATCH_SIZE,
                    "effective_batch_size": group_dro_batch_sampler.batch_size,
                    "eval_batch_size": EVAL_BATCH_SIZE,
                    "lr": LR,
                    "weight_decay": WEIGHT_DECAY,
                    "epochs": EPOCHS,
                    "group_dro_eta": GROUP_DRO_ETA,
                    "seed": SEED,
                }
            }, ckpt_path)

            val_pred_df.to_csv(out_dir / "best_val_predictions.csv", index=False)
            val_per_domain_df.to_csv(out_dir / "best_val_per_domain.csv", index=False)

            print(f"Saved new best checkpoint: {ckpt_path}")
            print(f"Best val EER: {best_val_eer:.4f}")

    pd.DataFrame(history).to_csv(out_dir / "training_history.csv", index=False)

    best_ckpt_path = out_dir / "best_group_dro.pt"

    if not best_ckpt_path.is_file():
        raise RuntimeError(
            f"No best Group DRO checkpoint was saved for {split_name}. "
            "Check validation metrics."
        )

    best_ckpt = torch.load(best_ckpt_path, map_location=DEVICE)

    model.load_state_dict(best_ckpt["model_state_dict"])
    model.to(DEVICE)
    model.float()

    test_result, test_per_domain_df, test_pred_df = evaluate(
        model,
        test_loader,
        split_name="test",
    )

    test_result["run"] = split_name
    test_result["split_dir"] = str(split_dir)
    test_result["best_epoch"] = best_epoch
    test_result["best_val_eer"] = best_val_eer
    test_result["best_val_loss"] = best_val_loss
    test_result["group_dro_eta"] = GROUP_DRO_ETA
    test_result["domain_label_batch_size"] = DOMAIN_LABEL_BATCH_SIZE
    test_result["effective_batch_size"] = group_dro_batch_sampler.batch_size
    test_result["initial_weights_path"] = INITIAL_WEIGHTS_PATH

    print(f"\n{split_name} | Final Group DRO Test Result:")
    print(test_result)

    print(f"\n{split_name} | Group DRO Test Per Domain:")
    print(test_per_domain_df)

    test_pred_df.to_csv(out_dir / "test_predictions.csv", index=False)
    test_per_domain_df.to_csv(out_dir / "test_per_domain.csv", index=False)

    with open(out_dir / "test_result.json", "w") as f:
        json.dump(test_result, f, indent=2)

    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    print(f"\nSaved Group DRO outputs to: {out_dir}")

    return test_result


# ============================================================
# Run selected splits
# ============================================================

split_dirs = discover_split_dirs(SPLITS_ROOT, SELECTED_SPLITS)

print(f"Found {len(split_dirs)} selected split directories under {SPLITS_ROOT}")
print(f"Selected splits: {', '.join(p.name for p in split_dirs)}")
print(f"Group DRO eta: {GROUP_DRO_ETA}")
print(f"Domain-label batch size: {DOMAIN_LABEL_BATCH_SIZE}")
print(f"Initial weights: {INITIAL_WEIGHTS_PATH}")
print(f"Results root: {RESULTS_ROOT}")

all_results = []

for split_dir in split_dirs:
    split_result = run_one_split(split_dir)
    all_results.append(split_result)

all_results_df = pd.DataFrame(all_results)
all_results_path = RESULTS_ROOT / "all_splits_test_results.csv"
all_results_df.to_csv(all_results_path, index=False)

summary = {
    "num_splits": len(all_results_df),
    "selected_splits": SELECTED_SPLITS,
    "mean_test_accuracy": float(all_results_df["accuracy"].mean()),
    "std_test_accuracy": float(all_results_df["accuracy"].std(ddof=0)),
    "mean_test_eer": float(all_results_df["eer"].mean()),
    "std_test_eer": float(all_results_df["eer"].std(ddof=0)),
    "group_dro_eta": GROUP_DRO_ETA,
    "domain_label_batch_size": DOMAIN_LABEL_BATCH_SIZE,
    "results_csv": str(all_results_path),
}

with open(RESULTS_ROOT / "all_splits_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nSelected Group DRO splits complete.")
print(all_results_df)
print("\nAggregate summary:")
print(summary)
print("Saved aggregate results:", all_results_path)

Found 4 selected split directories under /content/dg_splits_utterance_safe
Selected splits: split_00, split_09, split_14, split_12
Group DRO eta: 0.01
Domain-label batch size: 16
Initial weights: /content/rawnet_model_initial_weights.pth
Results root: /content/GROUP_DRO_results

Starting Group DRO run: split_00
Outputs: /content/GROUP_DRO_results/split_00

Loaded: /content/dg_splits_utterance_safe/split_00/main_train.csv
Rows before path check: 6268
Rows after path check:  6268
Label counts:
label
0    3134
1    3134
Name: count, dtype: int64
Paired domain counts:
paired_domain                                    label
tts_models_fr_css10_vits                         0        781
                                                 1        781
tts_models_fr_mai_tacotron2-DDC                  0        786
                                                 1        786
tts_models_multilingual_multi-dataset_bark       0        794
                                                 1        794
tt

split_00 | GroupDRO Epoch 01/10:   0%|          | 0/48 [00:00<?, ?it/s]


split_00 | Group DRO Epoch 01 done | Train obj: 0.5144 | Train mean loss: 0.5132 | Train worst-group loss: 0.6369 | Val loss: 2.5209 | Val acc: 0.5000 | Val EER: 0.3838 | Skipped batches: 0

Current Group DRO q weights:
{'tts_models_fr_css10_vits': 0.2393079698085785, 'tts_models_fr_mai_tacotron2-DDC': 0.24748772382736206, 'tts_models_multilingual_multi-dataset_bark': 0.25088921189308167, 'tts_models_multilingual_multi-dataset_xtts_v1.1': 0.26231512427330017}
Saved new best checkpoint: /content/GROUP_DRO_results/split_00/best_group_dro.pt
Best val EER: 0.3838


split_00 | GroupDRO Epoch 02/10:   0%|          | 0/48 [00:00<?, ?it/s]


split_00 | Group DRO Epoch 02 done | Train obj: 0.2601 | Train mean loss: 0.2591 | Train worst-group loss: 0.3773 | Val loss: 2.4832 | Val acc: 0.5122 | Val EER: 0.2709 | Skipped batches: 0

Current Group DRO q weights:
{'tts_models_fr_css10_vits': 0.23476892709732056, 'tts_models_fr_mai_tacotron2-DDC': 0.2440289705991745, 'tts_models_multilingual_multi-dataset_bark': 0.25739631056785583, 'tts_models_multilingual_multi-dataset_xtts_v1.1': 0.2638057768344879}
Saved new best checkpoint: /content/GROUP_DRO_results/split_00/best_group_dro.pt
Best val EER: 0.2709


...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x789c411dd260>
    Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
self._shutdown_workers()
Exception ignored in:   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
<function _MultiProcessingDataLoaderIter.__del__ at 0x789c411dd260>
    Traceback (most recent call last):
if w.is_alive():
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
      self._shutdown_workers() 
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
     if w.is_alive(): ^
 ^  ^^^^ ^Exception ignored in:   ^<function _MultiProcessingDataLoaderIter.__del__ at 0x789c411dd260>Exception ignored in:  
<function _MultiProcessingDataLoaderIter.__del__ at 0x789c411dd260>^Traceback (most 


split_12 | Group DRO Epoch 09 done | Train obj: 0.0879 | Train mean loss: 0.0829 | Train worst-group loss: 0.1749 | Val loss: 0.2212 | Val acc: 0.9138 | Val EER: 0.0583 | Skipped batches: 0

Current Group DRO q weights:
{'facebook_mms-tts-fra': 0.18812531232833862, 'griffin_lim': 0.3415951430797577, 'tts_models_fr_css10_vits': 0.21132421493530273, 'tts_models_multilingual_multi-dataset_xtts_v1.1': 0.25895532965660095}


split_12 | GroupDRO Epoch 10/10:   0%|          | 0/47 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x789c411dd260>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x789c411dd260>
^Traceback (most recent call last):
^Exception ignored in:   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
<function _MultiProcessingDataLoaderIter.__del__ at 0x789c411dd260>^Exception ignored in: 
^<function _MultiProcessingDataLoaderIter.__del__ at 0x789c41


split_12 | Group DRO Epoch 10 done | Train obj: 0.0796 | Train mean loss: 0.0741 | Train worst-group loss: 0.1449 | Val loss: 0.1121 | Val acc: 0.9607 | Val EER: 0.0393 | Skipped batches: 0

Current Group DRO q weights:
{'facebook_mms-tts-fra': 0.18526336550712585, 'griffin_lim': 0.34668922424316406, 'tts_models_fr_css10_vits': 0.20994721353054047, 'tts_models_multilingual_multi-dataset_xtts_v1.1': 0.2581001818180084}
Saved new best checkpoint: /content/GROUP_DRO_results/split_12/best_group_dro.pt
Best val EER: 0.0393

split_12 | Final Group DRO Test Result:
{'split': 'test', 'loss': 2.4783966956138612, 'accuracy': 0.594, 'eer': 0.3505, 'num_samples': 4000, 'run': 'split_12', 'split_dir': '/content/dg_splits_utterance_safe/split_12', 'best_epoch': 10, 'best_val_eer': 0.03929024081115336, 'best_val_loss': 0.11207272300168551, 'group_dro_eta': 0.01, 'domain_label_batch_size': 16, 'effective_batch_size': 128, 'initial_weights_path': '/content/rawnet_model_initial_weights.pth'}

split_12 

### Export GroupDRO results

In [ ]:
prepare_and_download('/content/GROUP_DRO_results', '/content/GROUP_DRO_results_clean', 'GROUP_DRO_results_clean.tar.gz')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Archived /content/GROUP_DRO_results (sans weights) and started download.


## CORAL

In [ ]:
import os
import random
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler

import torchaudio
from tqdm.auto import tqdm


# ============================================================
# Config
# ============================================================

SPLITS_ROOT = Path("/content/dg_splits_utterance_safe")
INITIAL_WEIGHTS_PATH = "/content/rawnet_model_initial_weights.pth"
RESULTS_ROOT = Path("/content/CORAL_results")
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

SELECTED_SPLITS = [
    "split_00",
    "split_09",
    "split_14",
    "split_12",
]

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TARGET_SR = 16000
CLIP_SECONDS = 4
CLIP_LEN = TARGET_SR * CLIP_SECONDS  # 64000

# For conditional CORAL, each domain/class needs enough samples
# to estimate covariance. 4 is better than 2 if memory allows.
DOMAIN_LABEL_BATCH_SIZE = 16
NUM_WORKERS = 4

EPOCHS = 10
LR = 1e-5
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 5.0

EVAL_BATCH_SIZE = 16

# CORAL-specific hyperparameter
CORAL_LAMBDA = 1.0

# Representation hook layer
HOOK_LAYER_NAME = "fc1_gru"


# ============================================================
# Reproducibility
# ============================================================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def seed_worker(worker_id):
    worker_seed = (torch.initial_seed() + worker_id) % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


seed_everything(SEED)


# ============================================================
# Validation helpers
# ============================================================

def require_file(path, description):
    if not os.path.isfile(path):
        raise FileNotFoundError(f"{description} not found: {path}")


def validate_split_dataframe(df, csv_path, split_name, require_domain=False):
    required_cols = {"audio_path", "label"}

    if require_domain:
        required_cols.add("paired_domain")

    missing = required_cols - set(df.columns)

    if missing:
        raise ValueError(f"Missing columns in {csv_path}: {sorted(missing)}")

    if len(df) == 0:
        raise ValueError(f"{split_name} split is empty: {csv_path}")

    invalid_labels = sorted(set(df["label"].dropna().unique()) - {0, 1})

    if invalid_labels:
        raise ValueError(
            f"{split_name} split has labels outside {{0, 1}}: {invalid_labels}"
        )

    if df["label"].nunique(dropna=True) != 2:
        raise ValueError(
            f"{split_name} split must contain both labels 0 and 1, "
            f"but found: {sorted(df['label'].dropna().unique())}"
        )


def validate_no_missing_paths(df, csv_path, split_name, out_dir):
    exists_mask = df["audio_path"].apply(
        lambda p: isinstance(p, str) and os.path.isfile(p)
    )

    missing_df = df.loc[~exists_mask].copy()

    if len(missing_df) == 0:
        return df.reset_index(drop=True)

    report_path = os.path.join(out_dir, f"{split_name}_missing_audio_paths.csv")
    missing_df.to_csv(report_path, index=False)

    raise FileNotFoundError(
        f"{split_name} split has {len(missing_df)} missing audio files. "
        f"Report saved to: {report_path}"
    )


def validate_loader_not_empty(loader, split_name):
    if len(loader.dataset) == 0:
        raise ValueError(f"{split_name} DataLoader has an empty dataset")


def discover_split_dirs(splits_root, selected_splits):
    split_dirs = [splits_root / split_name for split_name in selected_splits]

    missing_dirs = [str(p) for p in split_dirs if not p.is_dir()]
    if missing_dirs:
        raise FileNotFoundError(f"Selected split directories not found: {missing_dirs}")

    for split_dir in split_dirs:
        require_file(split_dir / "main_train.csv", f"{split_dir.name} train CSV")
        require_file(split_dir / "main_val.csv", f"{split_dir.name} val CSV")
        require_file(split_dir / "main_test.csv", f"{split_dir.name} test CSV")

    return split_dirs


# ============================================================
# Dataset
# ============================================================

class AudioCSVDataset(Dataset):
    def __init__(
        self,
        csv_path,
        target_sr=16000,
        clip_len=64000,
        train=True,
        split_name="split",
        require_domain=False,
        out_dir=None,
    ):
        require_file(csv_path, f"{split_name} CSV")

        self.df = pd.read_csv(csv_path)
        self.target_sr = target_sr
        self.clip_len = clip_len
        self.train = train
        self.split_name = split_name
        self.out_dir = out_dir

        validate_split_dataframe(
            self.df,
            csv_path,
            split_name,
            require_domain=require_domain,
        )

        before = len(self.df)
        self.df = validate_no_missing_paths(self.df, csv_path, split_name, out_dir)
        after = len(self.df)

        print(f"\nLoaded: {csv_path}")
        print(f"Rows before path check: {before}")
        print(f"Rows after path check:  {after}")
        print("Label counts:")
        print(self.df["label"].value_counts().sort_index())

        if "paired_domain" in self.df.columns:
            print("Paired domain counts:")
            print(self.df.groupby(["paired_domain", "label"]).size())

    def __len__(self):
        return len(self.df)

    def _load_audio(self, path):
        try:
            wav, sr = torchaudio.load(path)
        except Exception as exc:
            raise RuntimeError(f"Failed to load audio file: {path}") from exc

        if wav.numel() == 0:
            raise RuntimeError(f"Audio file is empty: {path}")

        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)

        if sr != self.target_sr:
            wav = torchaudio.functional.resample(wav, sr, self.target_sr)

        wav = wav.squeeze(0)

        if not torch.isfinite(wav).all():
            raise RuntimeError(f"Audio contains NaN or Inf values: {path}")

        max_val = wav.abs().max()
        if max_val > 0:
            wav = wav / max_val

        return wav

    def _crop_or_pad(self, wav):
        T = wav.numel()

        if T == self.clip_len:
            return wav

        if T > self.clip_len:
            if self.train:
                start = random.randint(0, T - self.clip_len)
            else:
                start = (T - self.clip_len) // 2

            return wav[start:start + self.clip_len]

        pad_len = self.clip_len - T
        return torch.nn.functional.pad(wav, (0, pad_len))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        wav = self._load_audio(row["audio_path"])
        wav = self._crop_or_pad(wav)

        label = torch.tensor(int(row["label"]), dtype=torch.long)

        item = {
            "waveform": wav,
            "label": label,
            "audio_path": row["audio_path"],
        }

        if "paired_domain" in self.df.columns:
            item["paired_domain"] = row["paired_domain"]

        if "spoof_generator" in self.df.columns:
            item["spoof_generator"] = row["spoof_generator"]

        if "utterance_id" in self.df.columns:
            item["utterance_id"] = row["utterance_id"]

        return item


def collate_fn(batch):
    if len(batch) == 0:
        raise ValueError("Received an empty batch")

    waveforms = torch.stack([x["waveform"] for x in batch], dim=0)
    labels = torch.stack([x["label"] for x in batch], dim=0)

    out = {
        "waveform": waveforms,
        "label": labels,
        "audio_path": [x["audio_path"] for x in batch],
    }

    for key in ["paired_domain", "spoof_generator", "utterance_id"]:
        if key in batch[0]:
            out[key] = [x[key] for x in batch]

    return out


# ============================================================
# Domain-label balanced batch sampler
# ============================================================

class DomainLabelBalancedBatchSampler(Sampler):
    """
    Creates batches containing all source domains.

    For each domain, each batch contains:
        DOMAIN_LABEL_BATCH_SIZE real samples
        DOMAIN_LABEL_BATCH_SIZE fake samples

    With 4 domains and DOMAIN_LABEL_BATCH_SIZE = 4:
        batch size = 4 * 2 * 4 = 32
    """

    def __init__(
        self,
        df,
        domain_col="paired_domain",
        label_col="label",
        per_label_per_domain=4,
        seed=42,
    ):
        self.df = df.reset_index(drop=True)
        self.domain_col = domain_col
        self.label_col = label_col
        self.per_label_per_domain = per_label_per_domain
        self.seed = seed
        self.epoch = 0

        self.domains = sorted(self.df[domain_col].unique().tolist())
        self.indices = {}

        for domain in self.domains:
            for label in [0, 1]:
                idxs = self.df.index[
                    (self.df[domain_col] == domain) &
                    (self.df[label_col] == label)
                ].tolist()

                if len(idxs) < self.per_label_per_domain:
                    raise ValueError(
                        f"Not enough samples for domain={domain}, label={label}. "
                        f"Found {len(idxs)}, need at least {self.per_label_per_domain}."
                    )

                self.indices[(domain, label)] = idxs

        self.num_batches = min(
            len(v) // self.per_label_per_domain
            for v in self.indices.values()
        )

        self.batch_size = len(self.domains) * 2 * self.per_label_per_domain

        print("\nConditional CORAL batch sampler:")
        print("Source domains:", self.domains)
        print("Samples per label per domain per batch:", self.per_label_per_domain)
        print("Effective batch size:", self.batch_size)
        print("Batches per epoch:", self.num_batches)

    def __len__(self):
        return self.num_batches

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __iter__(self):
        rng = random.Random(self.seed + self.epoch)

        shuffled = {}

        for key, idxs in self.indices.items():
            idxs_copy = idxs.copy()
            rng.shuffle(idxs_copy)
            shuffled[key] = idxs_copy

        pointers = {key: 0 for key in shuffled.keys()}

        for _ in range(self.num_batches):
            batch = []

            for domain in self.domains:
                for label in [0, 1]:
                    key = (domain, label)
                    start = pointers[key]
                    end = start + self.per_label_per_domain

                    batch.extend(shuffled[key][start:end])
                    pointers[key] = end

            rng.shuffle(batch)
            yield batch


def make_loaders(split_dir, out_dir):
    train_csv = split_dir / "main_train.csv"
    val_csv = split_dir / "main_val.csv"
    test_csv = split_dir / "main_test.csv"

    train_ds = AudioCSVDataset(
        train_csv,
        TARGET_SR,
        CLIP_LEN,
        train=True,
        split_name=f"{split_dir.name}_train",
        require_domain=True,
        out_dir=out_dir,
    )

    val_ds = AudioCSVDataset(
        val_csv,
        TARGET_SR,
        CLIP_LEN,
        train=False,
        split_name=f"{split_dir.name}_val",
        require_domain=False,
        out_dir=out_dir,
    )

    test_ds = AudioCSVDataset(
        test_csv,
        TARGET_SR,
        CLIP_LEN,
        train=False,
        split_name=f"{split_dir.name}_test",
        require_domain=False,
        out_dir=out_dir,
    )

    coral_batch_sampler = DomainLabelBalancedBatchSampler(
        train_ds.df,
        domain_col="paired_domain",
        label_col="label",
        per_label_per_domain=DOMAIN_LABEL_BATCH_SIZE,
        seed=SEED,
    )

    train_loader = DataLoader(
        train_ds,
        batch_sampler=coral_batch_sampler,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
        collate_fn=collate_fn,
        worker_init_fn=seed_worker,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
        collate_fn=collate_fn,
        worker_init_fn=seed_worker,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
        collate_fn=collate_fn,
        worker_init_fn=seed_worker,
    )

    validate_loader_not_empty(train_loader, f"{split_dir.name} train")
    validate_loader_not_empty(val_loader, f"{split_dir.name} val")
    validate_loader_not_empty(test_loader, f"{split_dir.name} test")

    return train_loader, val_loader, test_loader, coral_batch_sampler


# ============================================================
# Model setup
# ============================================================

if "model" not in globals():
    raise NameError(
        "Expected a pre-constructed variable named `model`, but none was found. "
        "Create/load your RawNet model object before running this CORAL cell."
    )

require_file(INITIAL_WEIGHTS_PATH, "Initial model weights")


def extract_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in ["model_state_dict", "state_dict", "model"]:
            if key in checkpoint and isinstance(checkpoint[key], dict):
                return checkpoint[key]

        if checkpoint and all(torch.is_tensor(v) for v in checkpoint.values()):
            return checkpoint

    raise ValueError(
        "Could not find a model state_dict in the initial weights file. "
        "Expected either a raw state_dict or a dict with model_state_dict/state_dict/model."
    )


def strip_module_prefix(state_dict):
    if not any(k.startswith("module.") for k in state_dict.keys()):
        return state_dict

    return {
        k.replace("module.", "", 1): v
        for k, v in state_dict.items()
    }


initial_checkpoint = torch.load(INITIAL_WEIGHTS_PATH, map_location="cpu")
initial_state_dict = strip_module_prefix(extract_state_dict(initial_checkpoint))

model = model.to(DEVICE)
model.float()

criterion = nn.CrossEntropyLoss(reduction="mean")


def reset_model_to_initial_weights(model):
    model.load_state_dict(initial_state_dict, strict=True)
    model.to(DEVICE)
    model.float()


# ============================================================
# Feature hook for CORAL
# ============================================================

class FeatureHook:
    def __init__(self, model, layer_name):
        self.model = model
        self.layer_name = layer_name
        self.features = None
        self.handle = None

        module = dict(model.named_modules()).get(layer_name, None)

        if module is None:
            available = list(dict(model.named_modules()).keys())
            raise ValueError(
                f"Layer '{layer_name}' not found. "
                f"Some available layers are: {available[:50]}"
            )

        self.handle = module.register_forward_hook(self._hook_fn)

    def _hook_fn(self, module, inputs, output):
        if isinstance(output, (tuple, list)):
            output = output[0]

        self.features = output

    def clear(self):
        self.features = None

    def remove(self):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None


feature_hook = None


def attach_feature_hook(model):
    global feature_hook

    if feature_hook is not None:
        feature_hook.remove()

    feature_hook = FeatureHook(model, HOOK_LAYER_NAME)

    print(f"Attached CORAL feature hook to layer: {HOOK_LAYER_NAME}")


def get_hooked_features():
    if feature_hook is None:
        raise RuntimeError("Feature hook has not been attached.")

    if feature_hook.features is None:
        raise RuntimeError("Feature hook did not capture features.")

    features = feature_hook.features

    # fc1_gru should give [B, 1024].
    # If a hooked layer returns [B, C, T], pool over time.
    if features.ndim == 3:
        features = features.mean(dim=-1)

    if features.ndim != 2:
        raise ValueError(f"Expected hooked features [B, D], got {features.shape}")

    if not torch.isfinite(features).all():
        raise RuntimeError("Hooked features contain NaN or Inf.")

    return features


def forward_model(model, waveforms, capture_features=False):
    """
    waveforms: [B, 64000]
    logits:    [B, 2]
    features:  [B, D] from fc1_gru when capture_features=True
    """
    if waveforms.ndim != 2 or waveforms.shape[1] != CLIP_LEN:
        raise ValueError(
            f"Expected waveforms shape [B, {CLIP_LEN}], got {waveforms.shape}"
        )

    if capture_features:
        feature_hook.clear()

    logits = model(waveforms)

    if logits.ndim != 2 or logits.shape[1] != 2:
        raise ValueError(f"Expected logits shape [B, 2], got {logits.shape}")

    if not torch.isfinite(logits).all():
        raise RuntimeError("Model produced NaN or Inf logits")

    if capture_features:
        features = get_hooked_features()
        return logits, features

    return logits


def compute_loss_and_scores(logits, labels):
    loss = criterion(logits, labels)
    scores = torch.softmax(logits, dim=1)[:, 1]
    return loss, scores


# ============================================================
# Conditional CORAL objective
# ============================================================

def covariance_matrix(features):
    """
    features: [N, D]
    returns:  [D, D]
    """
    n = features.shape[0]

    if n < 2:
        return None

    features = features - features.mean(dim=0, keepdim=True)
    cov = (features.T @ features) / (n - 1)

    return cov


def coral_pair_loss(features_a, features_b):
    """
    Standard CORAL covariance alignment loss:
        ||Cov(A) - Cov(B)||_F^2 / (4 * D^2)
    """
    cov_a = covariance_matrix(features_a)
    cov_b = covariance_matrix(features_b)

    if cov_a is None or cov_b is None:
        return None

    d = features_a.shape[1]
    loss = torch.mean((cov_a - cov_b) ** 2) / (4.0 * d * d)

    return loss


def conditional_coral_penalty(features, labels, domains):
    """
    Computes:
        CORAL(label=0 across domains) + CORAL(label=1 across domains)

    This prevents real and spoof distributions from being forced together.
    """
    unique_domains = sorted(set(domains))

    total_penalty = features.new_tensor(0.0)
    valid_pairs = 0

    label_penalties = {}

    for class_label in [0, 1]:
        class_domain_features = []

        for domain in unique_domains:
            mask_list = [
                (d == domain) for d in domains
            ]

            domain_mask = torch.tensor(
                mask_list,
                device=features.device,
                dtype=torch.bool,
            )

            class_mask = labels == class_label
            mask = domain_mask & class_mask

            f = features[mask]

            if f.shape[0] >= 2:
                class_domain_features.append((domain, f))

        class_penalty = features.new_tensor(0.0)
        class_pairs = 0

        for i in range(len(class_domain_features)):
            for j in range(i + 1, len(class_domain_features)):
                _, f_i = class_domain_features[i]
                _, f_j = class_domain_features[j]

                pair_loss = coral_pair_loss(f_i, f_j)

                if pair_loss is not None:
                    class_penalty = class_penalty + pair_loss
                    class_pairs += 1

        if class_pairs > 0:
            class_penalty = class_penalty / class_pairs
            total_penalty = total_penalty + class_penalty
            valid_pairs += 1

        label_penalties[class_label] = {
            "penalty": class_penalty,
            "num_pairs": class_pairs,
        }

    if valid_pairs > 0:
        total_penalty = total_penalty / valid_pairs

    return total_penalty, label_penalties


def compute_coral_objective(logits, features, labels, domains, coral_lambda):
    ce_loss = criterion(logits, labels)

    coral_penalty, label_penalties = conditional_coral_penalty(
        features,
        labels,
        domains,
    )

    objective = ce_loss + coral_lambda * coral_penalty

    return objective, ce_loss, coral_penalty, label_penalties


# ============================================================
# Metrics
# ============================================================

def compute_accuracy(labels, scores, threshold=0.5):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores).astype(float)

    if len(labels) == 0:
        return float("nan")

    preds = (scores >= threshold).astype(int)
    return float((preds == labels).mean())


def compute_eer(labels, scores):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores).astype(float)

    if len(labels) == 0:
        return float("nan")

    finite_mask = np.isfinite(scores)
    labels = labels[finite_mask]
    scores = scores[finite_mask]

    P = np.sum(labels == 1)
    N = np.sum(labels == 0)

    if P == 0 or N == 0:
        return float("nan")

    thresholds = np.r_[np.inf, np.unique(scores)[::-1], -np.inf]

    fars = []
    frrs = []

    for thr in thresholds:
        preds = (scores >= thr).astype(int)

        fp = np.sum((preds == 1) & (labels == 0))
        fn = np.sum((preds == 0) & (labels == 1))

        far = fp / N
        frr = fn / P

        fars.append(far)
        frrs.append(frr)

    fars = np.asarray(fars)
    frrs = np.asarray(frrs)

    idx = np.argmin(np.abs(fars - frrs))
    eer = (fars[idx] + frrs[idx]) / 2.0

    return float(eer)


@torch.no_grad()
def evaluate(model, loader, split_name="val"):
    model.eval()

    all_labels = []
    all_scores = []
    all_domains = []
    all_paths = []

    total_loss = 0.0
    total_n = 0

    for batch in loader:
        waveforms = batch["waveform"].to(DEVICE, non_blocking=True).float()
        labels = batch["label"].to(DEVICE, non_blocking=True).long()

        logits = forward_model(model, waveforms, capture_features=False)
        loss, scores = compute_loss_and_scores(logits, labels)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"{split_name} evaluation produced invalid loss: {loss.item()}"
            )

        bs = labels.shape[0]

        total_loss += loss.item() * bs
        total_n += bs

        all_labels.extend(labels.detach().cpu().numpy().tolist())
        all_scores.extend(scores.detach().cpu().numpy().tolist())
        all_paths.extend(batch["audio_path"])

        if "paired_domain" in batch:
            all_domains.extend(batch["paired_domain"])
        else:
            all_domains.extend(["unknown"] * bs)

    if total_n == 0:
        raise ValueError(f"{split_name} evaluation received zero samples")

    avg_loss = total_loss / total_n
    acc = compute_accuracy(all_labels, all_scores)
    eer = compute_eer(all_labels, all_scores)

    result = {
        "split": split_name,
        "loss": avg_loss,
        "accuracy": acc,
        "eer": eer,
        "num_samples": total_n,
    }

    pred_df = pd.DataFrame({
        "audio_path": all_paths,
        "label": all_labels,
        "score": all_scores,
        "paired_domain": all_domains,
    })

    per_domain_rows = []

    for domain, g in pred_df.groupby("paired_domain"):
        per_domain_rows.append({
            "split": split_name,
            "paired_domain": domain,
            "num_samples": len(g),
            "accuracy": compute_accuracy(g["label"].values, g["score"].values),
            "eer": compute_eer(g["label"].values, g["score"].values),
        })

    per_domain_df = pd.DataFrame(per_domain_rows)

    return result, per_domain_df, pred_df


# ============================================================
# Training
# ============================================================

def run_one_split(split_dir):
    split_name = split_dir.name
    out_dir = RESULTS_ROOT / split_name
    out_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 80)
    print(f"Starting Conditional CORAL run: {split_name}")
    print(f"Outputs: {out_dir}")
    print("=" * 80)

    seed_everything(SEED)
    reset_model_to_initial_weights(model)
    attach_feature_hook(model)

    train_loader, val_loader, test_loader, coral_batch_sampler = make_loaders(
        split_dir,
        out_dir,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    best_val_eer = float("inf")
    best_val_loss = float("inf")
    best_epoch = None
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        coral_batch_sampler.set_epoch(epoch)

        running_objective = 0.0
        running_ce = 0.0
        running_coral = 0.0
        running_real_coral = 0.0
        running_spoof_coral = 0.0
        running_n = 0
        skipped_batches = 0

        train_pbar = tqdm(
            train_loader,
            desc=f"{split_name} | CORAL Epoch {epoch:02d}/{EPOCHS}",
            leave=True,
            dynamic_ncols=True,
        )

        for batch in train_pbar:
            waveforms = batch["waveform"].to(DEVICE, non_blocking=True).float()
            labels = batch["label"].to(DEVICE, non_blocking=True).long()
            domains = batch["paired_domain"]

            optimizer.zero_grad(set_to_none=True)

            logits, features = forward_model(
                model,
                waveforms,
                capture_features=True,
            )

            objective, ce_loss, coral_penalty, label_penalties = compute_coral_objective(
                logits,
                features,
                labels,
                domains,
                coral_lambda=CORAL_LAMBDA,
            )

            if not torch.isfinite(objective):
                skipped_batches += 1
                warnings.warn(
                    f"Skipping batch due to invalid objective: {objective.item()}"
                )
                continue

            objective.backward()

            if GRAD_CLIP is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

            optimizer.step()

            bs = labels.shape[0]
            running_objective += objective.item() * bs
            running_ce += ce_loss.item() * bs
            running_coral += coral_penalty.item() * bs

            real_pen = label_penalties[0]["penalty"].item()
            spoof_pen = label_penalties[1]["penalty"].item()

            running_real_coral += real_pen * bs
            running_spoof_coral += spoof_pen * bs
            running_n += bs

            train_pbar.set_postfix({
                "obj": f"{running_objective / max(running_n, 1):.4f}",
                "ce": f"{running_ce / max(running_n, 1):.4f}",
                "coral": f"{running_coral / max(running_n, 1):.6f}",
                "skipped": skipped_batches,
            })

        if running_n == 0:
            raise RuntimeError(
                f"{split_name} epoch {epoch} completed with zero usable training samples. "
                f"Skipped batches: {skipped_batches}"
            )

        train_objective = running_objective / running_n
        train_ce = running_ce / running_n
        train_coral = running_coral / running_n
        train_real_coral = running_real_coral / running_n
        train_spoof_coral = running_spoof_coral / running_n

        val_result, val_per_domain_df, val_pred_df = evaluate(
            model,
            val_loader,
            split_name="val",
        )

        print(
            f"\n{split_name} | CORAL Epoch {epoch:02d} done | "
            f"Train obj: {train_objective:.4f} | "
            f"Train CE: {train_ce:.4f} | "
            f"Train CORAL: {train_coral:.8f} | "
            f"Real CORAL: {train_real_coral:.8f} | "
            f"Spoof CORAL: {train_spoof_coral:.8f} | "
            f"Val loss: {val_result['loss']:.4f} | "
            f"Val acc: {val_result['accuracy']:.4f} | "
            f"Val EER: {val_result['eer']:.4f} | "
            f"Skipped batches: {skipped_batches}\n"
        )

        epoch_record = {
            "split_dir": str(split_dir),
            "run": split_name,
            "epoch": epoch,
            "train_objective": train_objective,
            "train_ce": train_ce,
            "train_coral": train_coral,
            "train_real_coral": train_real_coral,
            "train_spoof_coral": train_spoof_coral,
            "val_loss": val_result["loss"],
            "val_accuracy": val_result["accuracy"],
            "val_eer": val_result["eer"],
            "val_num_samples": val_result["num_samples"],
            "skipped_batches": skipped_batches,
            "coral_lambda": CORAL_LAMBDA,
            "hook_layer": HOOK_LAYER_NAME,
        }

        history.append(epoch_record)

        val_eer = val_result["eer"]
        val_loss = val_result["loss"]

        if math.isfinite(val_eer):
            is_new_best = val_eer < best_val_eer
        else:
            is_new_best = not math.isfinite(best_val_eer) and val_loss < best_val_loss

        if is_new_best:
            best_val_eer = val_eer
            best_val_loss = val_loss
            best_epoch = epoch

            ckpt_path = out_dir / "best_coral.pt"

            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_eer": best_val_eer,
                "best_val_loss": best_val_loss,
                "config": {
                    "split_dir": str(split_dir),
                    "initial_weights_path": INITIAL_WEIGHTS_PATH,
                    "target_sr": TARGET_SR,
                    "clip_seconds": CLIP_SECONDS,
                    "clip_len": CLIP_LEN,
                    "domain_label_batch_size": DOMAIN_LABEL_BATCH_SIZE,
                    "effective_batch_size": coral_batch_sampler.batch_size,
                    "eval_batch_size": EVAL_BATCH_SIZE,
                    "lr": LR,
                    "weight_decay": WEIGHT_DECAY,
                    "epochs": EPOCHS,
                    "coral_lambda": CORAL_LAMBDA,
                    "hook_layer": HOOK_LAYER_NAME,
                    "seed": SEED,
                }
            }, ckpt_path)

            val_pred_df.to_csv(out_dir / "best_val_predictions.csv", index=False)
            val_per_domain_df.to_csv(out_dir / "best_val_per_domain.csv", index=False)

            print(f"Saved new best checkpoint: {ckpt_path}")
            print(f"Best val EER: {best_val_eer:.4f}")

    pd.DataFrame(history).to_csv(out_dir / "training_history.csv", index=False)

    best_ckpt_path = out_dir / "best_coral.pt"

    if not best_ckpt_path.is_file():
        raise RuntimeError(
            f"No best CORAL checkpoint was saved for {split_name}. "
            "Check validation metrics."
        )

    best_ckpt = torch.load(best_ckpt_path, map_location=DEVICE)

    model.load_state_dict(best_ckpt["model_state_dict"])
    model.to(DEVICE)
    model.float()

    test_result, test_per_domain_df, test_pred_df = evaluate(
        model,
        test_loader,
        split_name="test",
    )

    test_result["run"] = split_name
    test_result["split_dir"] = str(split_dir)
    test_result["best_epoch"] = best_epoch
    test_result["best_val_eer"] = best_val_eer
    test_result["best_val_loss"] = best_val_loss
    test_result["coral_lambda"] = CORAL_LAMBDA
    test_result["hook_layer"] = HOOK_LAYER_NAME
    test_result["domain_label_batch_size"] = DOMAIN_LABEL_BATCH_SIZE
    test_result["effective_batch_size"] = coral_batch_sampler.batch_size
    test_result["initial_weights_path"] = INITIAL_WEIGHTS_PATH

    print(f"\n{split_name} | Final Conditional CORAL Test Result:")
    print(test_result)

    print(f"\n{split_name} | Conditional CORAL Test Per Domain:")
    print(test_per_domain_df)

    test_pred_df.to_csv(out_dir / "test_predictions.csv", index=False)
    test_per_domain_df.to_csv(out_dir / "test_per_domain.csv", index=False)

    with open(out_dir / "test_result.json", "w") as f:
        json.dump(test_result, f, indent=2)

    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    print(f"\nSaved Conditional CORAL outputs to: {out_dir}")

    return test_result


# ============================================================
# Run selected splits
# ============================================================

split_dirs = discover_split_dirs(SPLITS_ROOT, SELECTED_SPLITS)

print(f"Found {len(split_dirs)} selected split directories under {SPLITS_ROOT}")
print(f"Selected splits: {', '.join(p.name for p in split_dirs)}")
print(f"CORAL lambda: {CORAL_LAMBDA}")
print(f"Hook layer: {HOOK_LAYER_NAME}")
print(f"Domain-label batch size: {DOMAIN_LABEL_BATCH_SIZE}")
print(f"Initial weights: {INITIAL_WEIGHTS_PATH}")
print(f"Results root: {RESULTS_ROOT}")

all_results = []

try:
    for split_dir in split_dirs:
        split_result = run_one_split(split_dir)
        all_results.append(split_result)

    all_results_df = pd.DataFrame(all_results)
    all_results_path = RESULTS_ROOT / "all_splits_test_results.csv"
    all_results_df.to_csv(all_results_path, index=False)

    summary = {
        "num_splits": len(all_results_df),
        "selected_splits": SELECTED_SPLITS,
        "mean_test_accuracy": float(all_results_df["accuracy"].mean()),
        "std_test_accuracy": float(all_results_df["accuracy"].std(ddof=0)),
        "mean_test_eer": float(all_results_df["eer"].mean()),
        "std_test_eer": float(all_results_df["eer"].std(ddof=0)),
        "coral_lambda": CORAL_LAMBDA,
        "hook_layer": HOOK_LAYER_NAME,
        "domain_label_batch_size": DOMAIN_LABEL_BATCH_SIZE,
        "results_csv": str(all_results_path),
    }

    with open(RESULTS_ROOT / "all_splits_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("\nSelected Conditional CORAL splits complete.")
    print(all_results_df)
    print("\nAggregate summary:")
    print(summary)
    print("Saved aggregate results:", all_results_path)

finally:
    if feature_hook is not None:
        feature_hook.remove()
        print("Removed CORAL feature hook.")

Found 4 selected split directories under /content/dg_splits_utterance_safe
Selected splits: split_00, split_09, split_14, split_12
CORAL lambda: 1.0
Hook layer: fc1_gru
Domain-label batch size: 16
Initial weights: /content/rawnet_model_initial_weights.pth
Results root: /content/CORAL_results

Starting Conditional CORAL run: split_00
Outputs: /content/CORAL_results/split_00
Attached CORAL feature hook to layer: fc1_gru

Loaded: /content/dg_splits_utterance_safe/split_00/main_train.csv
Rows before path check: 6268
Rows after path check:  6268
Label counts:
label
0    3134
1    3134
Name: count, dtype: int64
Paired domain counts:
paired_domain                                    label
tts_models_fr_css10_vits                         0        781
                                                 1        781
tts_models_fr_mai_tacotron2-DDC                  0        786
                                                 1        786
tts_models_multilingual_multi-dataset_bark       0        794


split_00 | CORAL Epoch 01/10:   0%|          | 0/48 [00:00<?, ?it/s]


split_00 | CORAL Epoch 01 done | Train obj: 0.5133 | Train CE: 0.5133 | Train CORAL: 0.00000000 | Real CORAL: 0.00000000 | Spoof CORAL: 0.00000000 | Val loss: 2.5154 | Val acc: 0.5000 | Val EER: 0.3838 | Skipped batches: 0

Saved new best checkpoint: /content/CORAL_results/split_00/best_coral.pt
Best val EER: 0.3838


split_00 | CORAL Epoch 02/10:   0%|          | 0/48 [00:00<?, ?it/s]


split_00 | CORAL Epoch 02 done | Train obj: 0.2594 | Train CE: 0.2594 | Train CORAL: 0.00000000 | Real CORAL: 0.00000000 | Spoof CORAL: 0.00000000 | Val loss: 2.4857 | Val acc: 0.5122 | Val EER: 0.2760 | Skipped batches: 0

Saved new best checkpoint: /content/CORAL_results/split_00/best_coral.pt
Best val EER: 0.2760


...


split_12 | CORAL Epoch 08 done | Train obj: 0.0911 | Train CE: 0.0911 | Train CORAL: 0.00000000 | Real CORAL: 0.00000000 | Spoof CORAL: 0.00000000 | Val loss: 0.1435 | Val acc: 0.9493 | Val EER: 0.0469 | Skipped batches: 0

Saved new best checkpoint: /content/CORAL_results/split_12/best_coral.pt
Best val EER: 0.0469


split_12 | CORAL Epoch 09/10:   0%|          | 0/47 [00:00<?, ?it/s]


split_12 | CORAL Epoch 09 done | Train obj: 0.0805 | Train CE: 0.0805 | Train CORAL: 0.00000000 | Real CORAL: 0.00000000 | Spoof CORAL: 0.00000000 | Val loss: 0.1786 | Val acc: 0.9309 | Val EER: 0.0469 | Skipped batches: 0



split_12 | CORAL Epoch 10/10:   0%|          | 0/47 [00:00<?, ?it/s]


split_12 | CORAL Epoch 10 done | Train obj: 0.0728 | Train CE: 0.0728 | Train CORAL: 0.00000000 | Real CORAL: 0.00000000 | Spoof CORAL: 0.00000000 | Val loss: 0.1301 | Val acc: 0.9544 | Val EER: 0.0418 | Skipped batches: 0

Saved new best checkpoint: /content/CORAL_results/split_12/best_coral.pt
Best val EER: 0.0418

split_12 | Final Conditional CORAL Test Result:
{'split': 'test', 'loss': 2.386813541173935, 'accuracy': 0.59225, 'eer': 0.3565, 'num_samples': 4000, 'run': 'split_12', 'split_dir': '/content/dg_splits_utterance_safe/split_12', 'best_epoch': 10, 'best_val_eer': 0.04182509505703422, 'best_val_loss': 0.13010432579907552, 'coral_lambda': 1.0, 'hook_layer': 'fc1_gru', 'domain_label_batch_size': 16, 'effective_batch_size': 128, 'initial_weights_path': '/content/rawnet_model_initial_weights.pth'}

split_12 | Conditional CORAL Test Per Domain:
  split                               paired_domain  num_samples  accuracy  \
0  test             tts_models_fr_mai_tacotron2-DDC        

### Export CORAL results

In [ ]:
prepare_and_download('/content/CORAL_results', '/content/CORAL_results_clean', 'CORAL_results_clean.tar.gz')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Archived /content/CORAL_results (sans weights) and started download.


## UMAP visualization

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchaudio
import matplotlib.pyplot as plt

try:
    import umap
except ImportError as exc:
    raise ImportError(
        "umap-learn is not installed. In Colab run: !pip install umap-learn"
    ) from exc


# ============================================================
# Config
# ============================================================

SPLITS_ROOT = Path("/content/dg_splits_utterance_safe")

RESULT_ROOTS = {
    "ERM": Path("/content/ERM_results"),
    "IRM": Path("/content/IRM_results"),
    "CORAL": Path("/content/CORAL_results"),
    "GroupDRO": Path("/content/GROUP_DRO_results"),
}

CHECKPOINT_NAMES = {
    "ERM": "best_erm.pt",
    "IRM": "best_irm.pt",
    "CORAL": "best_coral.pt",
    "GroupDRO": "best_group_dro.pt",
}

SELECTED_SPLITS = [
    "split_00",
    "split_09",
    "split_14",
    "split_12",
]

OUT_ROOT = Path("/content/DG_UMAP_results")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TARGET_SR = 16000
CLIP_SECONDS = 4
CLIP_LEN = TARGET_SR * CLIP_SECONDS

BATCH_SIZE = 64
NUM_WORKERS = 2

# RawNet final classifier:
#   fc1_gru: 1024 -> 1024
#   fc2_gru: 1024 -> 2
#
# The penultimate embedding requested for UMAP is the input to fc2_gru,
# i.e. the 1024-dimensional representation before the final classifier.
EMBED_LAYER_NAME = "fc2_gru"
EMBED_CAPTURE_MODE = "input"

UMAP_KWARGS = {
    "n_neighbors": 30,
    "min_dist": 0.1,
    "metric": "euclidean",
    "random_state": SEED,
}


# ============================================================
# Dataset
# ============================================================

class AudioCSVDataset(Dataset):
    def __init__(self, csv_path, target_sr=16000, clip_len=64000):
        self.df = pd.read_csv(csv_path)
        self.target_sr = target_sr
        self.clip_len = clip_len

        required_cols = {"audio_path", "label"}
        missing = required_cols - set(self.df.columns)
        if missing:
            raise ValueError(f"Missing columns in {csv_path}: {sorted(missing)}")

        if len(self.df) == 0:
            raise ValueError(f"Empty CSV: {csv_path}")

        exists_mask = self.df["audio_path"].apply(
            lambda p: isinstance(p, str) and os.path.isfile(p)
        )
        missing_paths = self.df.loc[~exists_mask]
        if len(missing_paths) > 0:
            raise FileNotFoundError(
                f"{csv_path} has {len(missing_paths)} missing audio paths. "
                f"First missing path: {missing_paths.iloc[0]['audio_path']}"
            )

        self.df = self.df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def _load_audio(self, path):
        wav, sr = torchaudio.load(path)

        if wav.numel() == 0:
            raise RuntimeError(f"Audio file is empty: {path}")

        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)

        if sr != self.target_sr:
            wav = torchaudio.functional.resample(wav, sr, self.target_sr)

        wav = wav.squeeze(0)

        if not torch.isfinite(wav).all():
            raise RuntimeError(f"Audio contains NaN or Inf values: {path}")

        max_val = wav.abs().max()
        if max_val > 0:
            wav = wav / max_val

        return wav

    def _center_crop_or_pad(self, wav):
        T = wav.numel()

        if T == self.clip_len:
            return wav

        if T > self.clip_len:
            start = (T - self.clip_len) // 2
            return wav[start:start + self.clip_len]

        pad_len = self.clip_len - T
        return torch.nn.functional.pad(wav, (0, pad_len))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        wav = self._load_audio(row["audio_path"])
        wav = self._center_crop_or_pad(wav)

        item = {
            "waveform": wav,
            "label": int(row["label"]),
            "audio_path": row["audio_path"],
            "paired_domain": row.get("paired_domain", "unknown"),
            "spoof_generator": row.get("spoof_generator", "unknown"),
            "utterance_id": row.get("utterance_id", "unknown"),
        }

        return item


def collate_fn(batch):
    return {
        "waveform": torch.stack([x["waveform"] for x in batch], dim=0),
        "label": torch.tensor([x["label"] for x in batch], dtype=torch.long),
        "audio_path": [x["audio_path"] for x in batch],
        "paired_domain": [x["paired_domain"] for x in batch],
        "spoof_generator": [x["spoof_generator"] for x in batch],
        "utterance_id": [x["utterance_id"] for x in batch],
    }


def make_test_loader(split_name):
    test_csv = SPLITS_ROOT / split_name / "main_test.csv"
    if not test_csv.is_file():
        raise FileNotFoundError(f"Missing test CSV: {test_csv}")

    ds = AudioCSVDataset(test_csv, TARGET_SR, CLIP_LEN)

    loader = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
        collate_fn=collate_fn,
    )

    return loader


# ============================================================
# Model / embedding helpers
# ============================================================

def extract_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in ["model_state_dict", "state_dict", "model"]:
            if key in checkpoint and isinstance(checkpoint[key], dict):
                return checkpoint[key]

        if checkpoint and all(torch.is_tensor(v) for v in checkpoint.values()):
            return checkpoint

    raise ValueError(
        "Could not find model weights. Expected a raw state_dict or a dict "
        "with model_state_dict/state_dict/model."
    )


def strip_module_prefix(state_dict):
    if not any(k.startswith("module.") for k in state_dict.keys()):
        return state_dict

    return {
        k.replace("module.", "", 1): v
        for k, v in state_dict.items()
    }


def load_checkpoint_into_model(model, checkpoint_path):
    if not os.path.isfile(checkpoint_path):
        raise FileNotFoundError(f"Missing checkpoint: {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    state_dict = strip_module_prefix(extract_state_dict(checkpoint))

    model.load_state_dict(state_dict, strict=True)
    model.to(DEVICE)
    model.float()
    model.eval()


def list_linear_layers(model):
    return [
        name for name, module in model.named_modules()
        if isinstance(module, nn.Linear)
    ]


def get_module_by_name(model, module_name):
    modules = dict(model.named_modules())
    if module_name not in modules:
        available = "\n".join(modules.keys())
        raise ValueError(
            f"Layer not found: {module_name}\nAvailable module names:\n{available}"
        )
    return modules[module_name]


def choose_embedding_layer(model):
    if EMBED_LAYER_NAME is not None:
        return EMBED_LAYER_NAME, get_module_by_name(model, EMBED_LAYER_NAME), EMBED_CAPTURE_MODE

    linear_layers = list_linear_layers(model)
    if not linear_layers:
        raise ValueError(
            "Could not find any nn.Linear layers. Set EMBED_LAYER_NAME manually."
        )

    final_linear_name = linear_layers[-1]
    final_linear = get_module_by_name(model, final_linear_name)

    # The input to the final classifier is usually the penultimate embedding.
    return final_linear_name, final_linear, "input"


@torch.no_grad()
def extract_embeddings(model, loader):
    layer_name, layer, capture_mode = choose_embedding_layer(model)
    captured = {}

    def hook_fn(module, inputs, output):
        if capture_mode == "input":
            emb = inputs[0]
        else:
            emb = output

        if isinstance(emb, (tuple, list)):
            emb = emb[0]

        captured["embedding"] = emb.detach()

    handle = layer.register_forward_hook(hook_fn)

    embeddings = []
    logits_all = []
    labels_all = []
    domains_all = []
    spoof_generators_all = []
    paths_all = []
    utts_all = []

    model.eval()

    try:
        for batch in loader:
            waveforms = batch["waveform"].to(DEVICE, non_blocking=True).float()
            logits = model(waveforms)

            if "embedding" not in captured:
                raise RuntimeError(
                    f"Embedding hook did not fire for layer: {layer_name}"
                )

            emb = captured.pop("embedding")
            emb = emb.detach().cpu()

            if emb.ndim > 2:
                emb = torch.flatten(emb, start_dim=1)

            embeddings.append(emb.numpy())
            logits_all.append(logits.detach().cpu().numpy())
            labels_all.extend(batch["label"].numpy().tolist())
            domains_all.extend(batch["paired_domain"])
            spoof_generators_all.extend(batch["spoof_generator"])
            paths_all.extend(batch["audio_path"])
            utts_all.extend(batch["utterance_id"])

    finally:
        handle.remove()

    embeddings = np.concatenate(embeddings, axis=0)
    logits_all = np.concatenate(logits_all, axis=0)
    scores = torch.softmax(torch.from_numpy(logits_all), dim=1)[:, 1].numpy()

    meta_df = pd.DataFrame({
        "audio_path": paths_all,
        "utterance_id": utts_all,
        "label": labels_all,
        "paired_domain": domains_all,
        "spoof_generator": spoof_generators_all,
        "score": scores,
    })

    return embeddings, meta_df, layer_name, capture_mode


# ============================================================
# Plotting
# ============================================================

def make_color_map(values, cmap_name="tab10"):
    unique_values = sorted(pd.Series(values).astype(str).unique().tolist())
    cmap = plt.get_cmap(cmap_name, max(len(unique_values), 1))
    return {
        value: cmap(i)
        for i, value in enumerate(unique_values)
    }


def plot_umap_panels(umap_frames, color_col, title, out_path):
    all_values = []
    for frame in umap_frames.values():
        all_values.extend(frame[color_col].astype(str).tolist())

    color_map = make_color_map(all_values)

    n_methods = len(umap_frames)
    fig, axes = plt.subplots(1, n_methods, figsize=(5 * n_methods, 4.5), squeeze=False)
    axes = axes[0]

    for ax, (method, frame) in zip(axes, umap_frames.items()):
        colors = frame[color_col].astype(str).map(color_map)

        ax.scatter(
            frame["umap_x"],
            frame["umap_y"],
            c=list(colors),
            s=8,
            alpha=0.75,
            linewidths=0,
        )

        ax.set_title(method)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlabel("")
        ax.set_ylabel("")

    handles = [
        plt.Line2D(
            [0], [0],
            marker="o",
            color="w",
            label=value,
            markerfacecolor=color,
            markersize=7,
        )
        for value, color in color_map.items()
    ]

    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=min(len(handles), 4),
        frameon=False,
    )

    fig.suptitle(title, y=0.98)
    fig.tight_layout(rect=[0, 0.12, 1, 0.93])
    fig.savefig(out_path, dpi=220)
    plt.close(fig)


def run_umap_for_split(split_name):
    split_out = OUT_ROOT / split_name
    split_out.mkdir(parents=True, exist_ok=True)

    loader = make_test_loader(split_name)
    umap_frames = {}
    embedding_info = {}

    print("\n" + "=" * 80)
    print(f"Running UMAP extraction for {split_name}")
    print("=" * 80)

    for method, result_root in RESULT_ROOTS.items():
        ckpt_path = result_root / split_name / CHECKPOINT_NAMES[method]

        print(f"\nLoading {method}: {ckpt_path}")
        load_checkpoint_into_model(model, ckpt_path)

        embeddings, meta_df, layer_name, capture_mode = extract_embeddings(model, loader)

        print(
            f"{method}: embeddings {embeddings.shape}, "
            f"layer={layer_name}, capture={capture_mode}"
        )

        reducer = umap.UMAP(**UMAP_KWARGS)
        coords = reducer.fit_transform(embeddings)

        frame = meta_df.copy()
        frame["method"] = method
        frame["umap_x"] = coords[:, 0]
        frame["umap_y"] = coords[:, 1]

        umap_frames[method] = frame
        embedding_info[method] = {
            "checkpoint": str(ckpt_path),
            "embedding_shape": list(embeddings.shape),
            "layer_name": layer_name,
            "capture_mode": capture_mode,
            "umap_kwargs": UMAP_KWARGS,
        }

        np.save(split_out / f"{method}_embeddings.npy", embeddings)
        frame.to_csv(split_out / f"{method}_umap_points.csv", index=False)

    with open(split_out / "embedding_info.json", "w") as f:
        json.dump(embedding_info, f, indent=2)

    plot_umap_panels(
        umap_frames,
        color_col="paired_domain",
        title=f"{split_name}: Held-out Test UMAP Colored by Generator",
        out_path=split_out / "umap_by_generator.png",
    )

    label_frames = {}
    for method, frame in umap_frames.items():
        label_frame = frame.copy()
        label_frame["label_name"] = label_frame["label"].map({
            0: "bonafide",
            1: "spoof",
        })
        label_frames[method] = label_frame

    plot_umap_panels(
        label_frames,
        color_col="label_name",
        title=f"{split_name}: Held-out Test UMAP Colored by Label",
        out_path=split_out / "umap_by_label.png",
    )

    print(f"\nSaved UMAP outputs to: {split_out}")


# ============================================================
# Run
# ============================================================

if "model" not in globals():
    raise NameError(
        "Expected a constructed RawNet variable named `model`. "
        "Create the same model architecture first, then run this cell/script."
    )

for split_name in SELECTED_SPLITS:
    run_umap_for_split(split_name)

print("\nAll UMAP figures complete.")
print("Output root:", OUT_ROOT)



Running UMAP extraction for split_00

Loading ERM: /content/ERM_results/split_00/best_erm.pt
ERM: embeddings (4000, 1024), layer=fc2_gru, capture=input


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(



Loading IRM: /content/IRM_results/split_00/best_irm.pt
IRM: embeddings (4000, 1024), layer=fc2_gru, capture=input


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(



Loading CORAL: /content/CORAL_results/split_00/best_coral.pt
CORAL: embeddings (4000, 1024), layer=fc2_gru, capture=input


...


Loading CORAL: /content/CORAL_results/split_12/best_coral.pt
CORAL: embeddings (4000, 1024), layer=fc2_gru, capture=input


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(



Loading GroupDRO: /content/GROUP_DRO_results/split_12/best_group_dro.pt
GroupDRO: embeddings (4000, 1024), layer=fc2_gru, capture=input


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(



Saved UMAP outputs to: /content/DG_UMAP_results/split_12

All UMAP figures complete.
Output root: /content/DG_UMAP_results


### Export UMAP results

In [ ]:
import os
from google.colab import files

directory_to_archive = "/content/DG_UMAP_results"
archive_name = "DG_UMAP_results.tar.gz"

# Create the tar.gz archive
!tar -czvf {archive_name} {directory_to_archive}

# Download the archive
files.download(archive_name)

print(f"Archived '{directory_to_archive}' and started download as '{archive_name}'.")

tar: Removing leading `/' from member names
/content/DG_UMAP_results/
/content/DG_UMAP_results/split_14/
/content/DG_UMAP_results/split_14/GroupDRO_umap_points.csv
/content/DG_UMAP_results/split_14/embedding_info.json
/content/DG_UMAP_results/split_14/GroupDRO_embeddings.npy
/content/DG_UMAP_results/split_14/CORAL_umap_points.csv
/content/DG_UMAP_results/split_14/umap_by_label.png
/content/DG_UMAP_results/split_14/CORAL_embeddings.npy
/content/DG_UMAP_results/split_14/IRM_embeddings.npy
/content/DG_UMAP_results/split_14/umap_by_generator.png
/content/DG_UMAP_results/split_14/ERM_umap_points.csv
/content/DG_UMAP_results/split_14/ERM_embeddings.npy
/content/DG_UMAP_results/split_14/IRM_umap_points.csv
/content/DG_UMAP_results/split_00/
/content/DG_UMAP_results/split_00/GroupDRO_umap_points.csv
/content/DG_UMAP_results/split_00/embedding_info.json
/content/DG_UMAP_results/split_00/GroupDRO_embeddings.npy
/content/DG_UMAP_results/split_00/CORAL_umap_points.csv
/content/DG_UMAP_results/spli

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Archived '/content/DG_UMAP_results' and started download as 'DG_UMAP_results.tar.gz'.


## Per-domain test results

In [ ]:
import pandas as pd
from pathlib import Path

RESULT_ROOTS = {
    "ERM": Path("/content/ERM_results"),
    "IRM": Path("/content/IRM_results"),
    "CORAL": Path("/content/CORAL_results"),
    "GroupDRO": Path("/content/GROUP_DRO_results"),
}

SELECTED_SPLITS = [
    "split_00",
    "split_09",
    "split_14",
    "split_12",
]

for method, root in RESULT_ROOTS.items():
    print("\n" + "=" * 80)
    print(method)
    print("=" * 80)

    for split in SELECTED_SPLITS:
        csv_path = root / split / "test_per_domain.csv"

        print(f"\n{split}: {csv_path}")

        if not csv_path.exists():
            print("Missing file")
            continue

        df = pd.read_csv(csv_path)
        print(df.to_string(index=False))



ERM

split_00: /content/ERM_results/split_00/test_per_domain.csv
split        paired_domain  num_samples  accuracy   eer
 test facebook_mms-tts-fra         2000    0.8060 0.114
 test          griffin_lim         2000    0.5215 0.430

split_09: /content/ERM_results/split_09/test_per_domain.csv
split                   paired_domain  num_samples  accuracy   eer
 test        tts_models_fr_css10_vits         2000    0.4605 0.524
 test tts_models_fr_mai_tacotron2-DDC         2000    0.5565 0.388

split_14: /content/ERM_results/split_14/test_per_domain.csv
split                                   paired_domain  num_samples  accuracy   eer
 test      tts_models_multilingual_multi-dataset_bark         2000    0.6415 0.333
 test tts_models_multilingual_multi-dataset_xtts_v1.1         2000    0.5190 0.367

split_12: /content/ERM_results/split_12/test_per_domain.csv
split                              paired_domain  num_samples  accuracy   eer
 test            tts_models_fr_mai_tacotron2-DDC       